In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

import torch
import torch.nn as nn
import torch.nn.functional as F

import itertools
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from xgboost import XGBRegressor, XGBClassifier
from lightgbm import LGBMRegressor, LGBMClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import StandardScaler, LabelEncoder
import joblib
import re
from typing import Dict, List, Tuple, Any

import glob
from korean_lunar_calendar import KoreanLunarCalendar

# Jupyter notebook에서 출력 제한 해제
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
# ----------------------------
# Logging
# ----------------------------
import logging

logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s %(levelname)s %(message)s",
    handlers=[
        logging.FileHandler("debug.log", encoding="utf-8"),  # 파일: 무제한
        logging.StreamHandler()                              # 노트북 셀: 요약만
    ]
)
# ----------------------------
# Data preprocessing
# ----------------------------
def load_and_preprocess_data(file_path):
    """기본 데이터 로딩 및 전처리"""
    df = pd.read_csv(file_path)
    
    # 업장명과 메뉴명 분리
    df[['store', 'menu']] = df['영업장명_메뉴명'].str.split('_', expand=True, n=1)
    df['date'] = pd.to_datetime(df['영업일자'])
    df['sales'] = df['매출수량']
    
    # 시간 관련 피처 생성
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['day_of_week'] = df['date'].dt.dayofweek  # 0=월요일
    df['is_weekend'] = df['day_of_week'].isin([5, 6])  # 토, 일
    df['is_holiday_season'] = df['month'].isin([7, 8, 12, 1])  # 성수기
    df = generate_combined_holiday_list(df, solar_md_holidays, lunar_solar_dates)
    df = filter_all_menus_by_leading_zeros(
        df,
        min_zero_days=90,
        apply_to_stores=['담하','라그로타','미라시아' ]  # 여기에 대상 업장명만 나열
    )

    return df[['date', 'store', 'menu', 'sales', 'year', 'month', 'day_of_week', 'is_weekend', 'is_holiday_season','is_holiday']]

def create_time_features(df):
    df = df.sort_values(['menu','date']).copy()
    g = df.groupby('menu', group_keys=False)

    # lag: NaN -> 0, 그리고 결측 플래그 추가
    for lag in [1,7,14,28]:
        col = f'lag_{lag}'
        df[col] = g['sales'].shift(lag)
        df[f'{col}_missing'] = df[col].isna().astype('int8')
        df[col] = df[col].fillna(0.0)
    # 시계열 분할 시 미래 정보 차단
    for w in [3,7,14]:
        df[f'ma_{w}'] = g['sales'].shift(1).transform(lambda s: s.rolling(w, min_periods=1).mean())

    # 표준편차: 관측 1개일 때 0이 되도록 ddof=0, min_periods=1
    df['rolling_std_7'] = g['sales'].transform(lambda s: s.rolling(7, min_periods=1).std(ddof=0)).fillna(0.0)
    return df

# 업장 카테고리 정의
STORE_CATEGORIES = {
    'damha_individual': ['담하'],
    'mirasia_individual': ['미라시아'],
    'main_restaurants': ['포레스트릿', '카페테리아', '화담숲주막', '화담숲카페'],
    'regular_outlets': ['느티나무 셀프BBQ'],
    'special_venues': ['연회장', '라그로타']
}
SEQUENCELENGTH = 28
PREDICT = 7
def split_data(df, store: str):
    return df[df['store'].astype(str).str.contains(store, na=False)].copy()
def remove_leading_zeros_before_sales(df, min_zero_days=90):
    """
    매출 시작 전 연속 0이 일정 기간 이상이면, 그 전 구간 제거
    (단일 메뉴-업장 그룹 DataFrame을 가정)
    """
    sales_started = df['매출수량'] > 0
    if not sales_started.any():
        return df  # 매출이 전혀 없는 경우 그대로 반환

    first_sale_idx = sales_started.idxmax()

    # 매출 시작 전 구간이 충분히 긴 0으로 구성되어 있다면 제거
    df_before = df.loc[:first_sale_idx - 1]
    if len(df_before) >= min_zero_days and (df_before['매출수량'] == 0).all():
        return df.loc[first_sale_idx:]  # 매출 시작부터 반환
    else:
        return df  # 그대로 반환
def _extract_store_name(g: pd.DataFrame) -> str:
    """
    그룹 g에서 업장명 추출:
    - '영업장명' 컬럼이 있으면 그 값을 사용
    - 없으면 '영업장명_메뉴명'에서 첫 '_' 앞을 업장명으로 간주
    """
    if '영업장명' in g.columns:
        return str(g['영업장명'].iloc[0])
    # '영업장명_메뉴명'이 "업장명_메뉴명" 형태라고 가정
    full = str(g['영업장명_메뉴명'].iloc[0])
    return full.split('_', 1)[0]  # '_'가 여러 개여도 첫 구분만 사용


def filter_all_menus_by_leading_zeros(
    train_df: pd.DataFrame,
    min_zero_days: int = 90,
    apply_to_stores: list[str] | None = None,
    exclude_stores: list[str] | None = None,
    group_col: str = '영업장명_메뉴명',
) -> pd.DataFrame:
    """
    모든 메뉴-업장 그룹에 대해 remove_leading_zeros_before_sales를 적용하되,
    특정 업장에만(또는 특정 업장은 제외하고) 적용할 수 있도록 확장.

    Parameters
    ----------
    train_df : 전체 데이터프레임
    min_zero_days : 매출 시작 전 연속 0 최소 일수
    apply_to_stores : 적용 대상 업장명 리스트 (None이면 전 업장 대상)
    exclude_stores : 적용 제외 업장명 리스트 (None이면 제외 없음)
    group_col : 그룹화 기준 컬럼명 (기본: '영업장명_메뉴명')
    """
    parts = []
    apply_set   = set(apply_to_stores) if apply_to_stores is not None else None
    exclude_set = set(exclude_stores)  if exclude_stores  is not None else set()

    # 기존 순서 보존 원하면 sort=False 유지
    for _, g in train_df.groupby(group_col, sort=False):
        store = _extract_store_name(g)

        # 적용 여부 결정
        apply_flag = True
        if apply_set is not None:
            apply_flag = (store in apply_set)
        if store in exclude_set:
            apply_flag = False

        if apply_flag:
            parts.append(remove_leading_zeros_before_sales(g, min_zero_days))
        else:
            parts.append(g)

    if parts:
        return pd.concat(parts, ignore_index=True)
    return train_df.reset_index(drop=True)
def get_lunar_to_solar(years, lunar_month, lunar_day, span=1):
    calendar = KoreanLunarCalendar()
    dates = []
    for year in years:
        for offset in range(-span, span+1):
            try:
                calendar.setLunar(year, lunar_month, lunar_day + offset, False)
                dates.append(calendar.SolarIsoFormat())
            except:
                pass
    return dates

years = [2023, 2024, 2025]
lunar_solar_dates = []
lunar_solar_dates += get_lunar_to_solar(years, 1, 1, span=1)   # 설 ±1
lunar_solar_dates += get_lunar_to_solar(years, 8, 15, span=1)  # 추석 ±1

solar_md_holidays = [
    (1, 1), (3, 1), (5, 5), (6, 6), (8, 15), (10, 3), (10, 9), (12, 25)
]
def generate_combined_holiday_list(df, solar_md_list, lunar_solar_list):
    df = df.copy()
    df['영업일자'] = pd.to_datetime(df['영업일자'])
    df['is_solar_holiday'] = df['영업일자'].apply(lambda x: (x.month, x.day) in solar_md_list)
    lunar_set = set(pd.to_datetime(lunar_solar_list))
    df['is_lunar_holiday'] = df['영업일자'].isin(lunar_set)
    df['is_holiday'] = (df['is_solar_holiday'] | df['is_lunar_holiday']).astype(int)
    return df.drop(columns=['is_solar_holiday', 'is_lunar_holiday'])

def split_data_by_category(df):
    """카테고리별로 데이터 분리"""
    category_data = {}
    
    for category, stores in STORE_CATEGORIES.items():
        category_df = df[df['store'].isin(stores)].copy()
        category_data[category] = category_df
        print(f"{category}: {len(category_df)} records, {len(stores)} stores")
    
    return category_data
def create_sequences_for_prediction(df, seq_length=28, pred_length=7, min_data_points=35):
    """
    28일 데이터로 다음 7일을 예측하는 시퀀스 생성
    
    Parameters:
    - df: 전처리된 데이터프레임
    - seq_length: 입력 시퀀스 길이 (28일)
    - pred_length: 예측 길이 (7일)
    - min_data_points: 최소 데이터 포인트 (35일 = 28+7)
    """
    sequences = []
    targets = []
    metadata = []  # 날짜, 업장, 메뉴 정보 저장
    
    # 업장-메뉴 조합별로 시계열 생성
    for store in df['store'].unique():
        store_data = df[df['store'] == store]
        
        for menu in store_data['menu'].unique():
            menu_data = store_data[store_data['menu'] == menu].sort_values('date')
            
            # 충분한 데이터가 있는 경우만 시퀀스 생성
            if len(menu_data) >= min_data_points:
                X, y, meta = create_single_menu_sequences(
                    menu_data, store, menu, seq_length, pred_length
                )
                sequences.extend(X)
                targets.extend(y)
                metadata.extend(meta)
    
    return np.array(sequences), np.array(targets), metadata
def create_single_menu_sequences(menu_data, store, menu, seq_length, pred_length):
    """단일 메뉴에 대한 시퀀스 생성"""
    X, y, metadata = [], [], []
    
    for i in range(len(menu_data) - seq_length - pred_length + 1):
        # 입력 시퀀스 (28일)
        seq_data = menu_data.iloc[i:i+seq_length]
        
        # 타겟 시퀀스 (7일)  
        target_data = menu_data.iloc[i+seq_length:i+seq_length+pred_length]
        
        # 피처 벡터 생성 (매출 + 시간 피처)
        seq_features = np.column_stack([
            seq_data['sales'].values,
            seq_data['day_of_week'].values,
            seq_data['is_weekend'].astype(int).values,
            seq_data['is_holiday_season'].astype(int).values
        ])
        
        X.append(seq_features)
        y.append(target_data['sales'].values)
        
        # 메타데이터 (디버깅/분석용)
        metadata.append({
            'store': store,
            'menu': menu,
            'seq_start_date': seq_data['date'].iloc[0],
            'pred_start_date': target_data['date'].iloc[0]
        })
    
    return X, y, metadata

class CategorySpecificPreprocessor:
    def __init__(self):
        self.damha_processor = DamhaPreprocessor()
        self.mirasia_processor = MirasiaPreprocessor()
    
    def preprocess_by_category(self, df, category):
        """카테고리별 특화 전처리 실행"""
        
        if category == 'damha_individual':
            return self.damha_processor.preprocess_damha_data(df)
            
        elif category == 'mirasia_individual':
            return self.mirasia_processor.preprocess_mirasia_data(df)
            
        elif category == 'main_restaurants':
            return self._preprocess_main_restaurants(df)
            
        elif category == 'regular_outlets':
            return self._preprocess_regular_outlets(df)
            
        elif category == 'special_venues':
            return self._preprocess_special_venues(df)
        
        else:
            return df
    
    def _preprocess_main_restaurants(self, df):
        """메인 레스토랑 전처리 (포레스트릿, 카페테리아, 화담숲주막, 화담숲카페)"""
        main_stores = ['포레스트릿', '카페테리아', '화담숲주막', '화담숲카페']
        main_df = df[df['store'].isin(main_stores)].copy()
        
        # 업장별 매출 규모 정규화를 위한 스케일 팩터
        store_scale = {
            '포레스트릿': 'large',    # 평균 47.84
            '화담숲주막': 'large',    # 평균 34.38  
            '화담숲카페': 'medium',   # 평균 23.55
            '카페테리아': 'medium'    # 평균 18.86
        }
        main_df['store_scale'] = main_df['store'].map(store_scale)
        
        # 고매출 업장 플래그
        main_df['is_high_volume'] = main_df['store'].isin(['포레스트릿', '화담숲주막'])
        
        return main_df

# ----------------------------
# Damha class
# ----------------------------

class DamhaPreprocessor:
    def __init__(self):
        # 담하 메뉴 카테고리 분류 (분석 결과 기반)
        self.menu_categories = {
            'core_rice_dishes': ['공깃밥', '생목살 김치찌개', '한우 차돌박이 된장찌개'],  # 저 0매출 비율, 높은 매출
            'main_dishes': ['담하 한우 불고기', '한우 우거지 국밥', '한우 떡갈비 정식', '황태해장국'],
            'group_menus': ['(단체) 황태해장국 3/27까지'],  # 높은 0매출 비율
            'set_menus': ['담하 한우 불고기 정식'],  # 중간 0매출 비율
            'others': []  # 나머지 메뉴들
        }
        
    def categorize_menu(self, menu_name):
        """메뉴를 카테고리별로 분류"""
        for category, menus in self.menu_categories.items():
            if menu_name in menus:
                return category
        return 'others'
    
    def preprocess_damha_data(self, df):
        """담하 전용 전처리"""
        damha_df = df[df['store'] == '담하'].copy()
        
        # 1. 메뉴 카테고리 추가
        damha_df['menu_category'] = damha_df['menu'].apply(self.categorize_menu)
        
        # 2. 단체메뉴 플래그
        damha_df['is_group_menu'] = damha_df['menu'].str.contains('단체', na=False)
        
        # 3. 정식메뉴 플래그  
        damha_df['is_set_menu'] = damha_df['menu'].str.contains('정식', na=False)
        
        # 4. 핵심메뉴 플래그 (높은 매출, 낮은 0비율)
        core_menus = ['공깃밥', '담하 한우 불고기', '생목살 김치찌개', '한우 차돌박이 된장찌개']
        damha_df['is_core_menu'] = damha_df['menu'].isin(core_menus)
        
        # 5. 매출 임계값 기반 이진 분류 (Zero-inflation 대응)
        damha_df['has_sales'] = (damha_df['sales'] > 0).astype(int)
        
        # 6. 메뉴별 과거 평균 매출 (이동평균)
        damha_df = self._add_menu_rolling_features(damha_df)
        
        return damha_df
    
    def _add_menu_rolling_features(self, df):
        """메뉴별 이동평균 및 통계 피처 추가"""
        df = df.sort_values(['menu', 'date'])
        
        # 메뉴별 7일, 14일 이동평균
        df['menu_sales_7d_avg'] = df.groupby('menu')['sales'].transform(
            lambda x: x.rolling(window=7, min_periods=1).mean()
        )
        df['menu_sales_14d_avg'] = df.groupby('menu')['sales'].transform(
            lambda x: x.rolling(window=14, min_periods=1).mean()
        )
        
        # 메뉴별 요일 평균 매출
        df['menu_dow_avg'] = df.groupby(['menu', 'day_of_week'])['sales'].transform('mean')
        
        return df

class DamhaMLModel:
    def __init__(self, seq_length: int = SEQUENCELENGTH, top_k_menus: int = 30,
                 use_zero_gate: bool = False, zero_gate_thr: float = 0.15):
        # 설정
        self.seq_length = seq_length
        self.top_k_menus = top_k_menus
        self.use_zero_gate = use_zero_gate
        self.zero_gate_thr = zero_gate_thr
        self._is_fitted = False

        # Zero-inflation 분류기 (하이퍼파라미터는 fit 때 보정 가능)
        self.zero_classifier = XGBClassifier(
            n_estimators=400,
            max_depth=4,
            learning_rate=0.1,
            min_child_weight=1.0,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            n_jobs=-1,
            verbosity=0
        )

        # 일자별 회귀기 컨테이너
        self.daily_regressors: List[XGBRegressor] = []
        self.menu_models = {}
        self.core_menus = ['공깃밥', '담하 한우 불고기', '생목살 김치찌개']

        # 인코딩 스키마
        self.top_menus_ = None
        self.feature_cols_ = None

    def create_damha_features(self, df, mode='train'):
        seq_length = self.seq_length
        features_list, targets_list = [], []
        
        for menu in df['menu'].unique():
            md = df[df['menu'] == menu].sort_values('date').reset_index(drop=True)
            
            # 모드별 조건 분기
            if mode == 'predict':
                if len(md) < seq_length:  # 예측 모드: 28일만 필요
                    continue
            else:  # train 모드
                if len(md) < seq_length + PREDICT:  # 훈련 모드: 35일 필요
                    continue
            
            dow = md['day_of_week'].to_numpy()
            month = md['month'].to_numpy()
            hol = md.get('is_holiday', pd.Series(False, index=md.index)).astype(int).to_numpy()
            
            if mode == 'predict':
                # 예측 모드: 마지막 28일로 1개 피처만 생성
                seq = md.tail(seq_length)
                
                # 미래 7일 정보는 패턴으로 추정
                last_date = seq['date'].iloc[-1]
                future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=7)
                future_dow = future_dates.dayofweek.to_numpy()
                future_month = future_dates.month.to_numpy()
                future_holiday = np.zeros(7, dtype=int)  # 간단히 0으로 가정
                
                f = {
                    'sales_mean': float(seq['sales'].mean()),
                    'sales_std': float(seq['sales'].std()),
                    'sales_max': float(seq['sales'].max()),
                    'sales_min': float(seq['sales'].min()),
                    'sales_last_7d_mean': float(seq['sales'].tail(7).mean()),
                    'sales_last_3d_mean': float(seq['sales'].tail(3).mean()),
                    'zero_sales_ratio': float((seq['sales'] == 0).mean()),
                    'consecutive_zeros': float(self._consecutive_zeros(seq['sales'])),
                    'days_since_last_sale': float(self._days_since_last_sale(seq['sales'])),
                    'weekend_sales_ratio': float(
                        (seq.loc[seq['is_weekend'], 'sales'].mean() or 0.0) / (seq['sales'].mean() + 1e-8)
                    ),
                    'friday_avg': float(seq.loc[seq['day_of_week']==4, 'sales'].mean()),
                    'saturday_avg': float(seq.loc[seq['day_of_week']==5, 'sales'].mean()),
                    'sunday_avg': float(seq.loc[seq['day_of_week']==6, 'sales'].mean()),
                    'is_core_menu': int(menu in self.core_menus),
                    'is_group_menu': int('단체' in menu),
                    'is_set_menu': int('정식' in menu),
                    'sales_trend': float(self._calculate_trend(seq['sales'])),
                    'recent_growth': float(seq['sales'].tail(7).mean() - seq['sales'].head(7).mean()),
                    'menu': menu
                }
                
                # 미래 일자별 피처 추가
                for h in range(7):
                    f[f'target_dow_{h}'] = int(future_dow[h])
                    f[f'target_month_{h}'] = int(future_month[h])
                    f[f'target_holiday_{h}'] = int(future_holiday[h])
                
                features_list.append(f)
                
            else:  # train 모드 - 기존 로직
                for i in range(len(md) - seq_length - PREDICT + 1):
                    seq = md.iloc[i:i+seq_length]
                    tgt = md.iloc[i+seq_length:i+seq_length+7]
                    
                    f = {
                        'sales_mean': float(seq['sales'].mean()),
                        # ... 기존 피처들 동일 ...
                        'menu': menu
                    }
                    
                    # 미래 일자별 피처(요일/월/휴일) 추가
                    t_idx = np.arange(i + seq_length, i + seq_length + 7)
                    for h in range(7):
                        f[f'target_dow_{h}'] = int(dow[t_idx[h]])
                        f[f'target_month_{h}'] = int(month[t_idx[h]])
                        f[f'target_holiday_{h}'] = int(hol[t_idx[h]])
                    
                    features_list.append(f)
                    targets_list.append(tgt['sales'].to_numpy())
        
        X = pd.DataFrame(features_list).fillna(0.0)
        y = np.array(targets_list) if targets_list else np.empty((0, 7))
        return X, y

    def _consecutive_zeros(self, s):
        z = (s == 0).astype(int).tolist()
        return max((len(list(g)) for k, g in itertools.groupby(z) if k), default=0)

    def _days_since_last_sale(self, s):
        nz = np.where(np.array(s) > 0)[0]
        return len(s) if len(nz) == 0 else (len(s) - 1 - nz[-1])

    def _calculate_trend(self, s):
        if len(s) < 2: return 0.0
        x = np.arange(len(s))
        try:
            return float(np.polyfit(x, np.array(s, dtype=float), 1)[0])
        except Exception:
            return 0.0

    # ---------- Encoding ----------

    def _encode_features(self, X: pd.DataFrame, fit: bool = False) -> pd.DataFrame:
        """menu 원-핫 + 숫자형만 유지 + 스키마 고정(학습/예측 일관성)"""
        # 0) 입력 보정
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X)
        Xe = X.copy()

        # 1) top_k 메뉴 수 결정 (self.top_k_menus 또는 self.cfg.top_k_menus 사용)
        top_k = getattr(self, 'top_k_menus', None)
        if top_k is None and hasattr(self, 'cfg'):
            top_k = getattr(self.cfg, 'top_k_menus', 30)
        if top_k is None:
            top_k = 30

        # 2) top_menus_ 초기화 (fit 시점 또는 비어있을 때)
        if fit or getattr(self, 'top_menus_', None) is None:
            if 'menu' not in Xe.columns:
                raise ValueError("X must include 'menu' at fit-time.")
            self.top_menus_ = Xe['menu'].astype(str).value_counts().head(top_k).index.tolist()
        else:
            # predict 시점인데 top_menus_가 없다면, fit이 선행되지 않은 것
            if getattr(self, 'top_menus_', None) is None:
                raise RuntimeError("Model not fitted: top_menus_ is not set. Call fit() first.")

        # 3) 메뉴명 -> 안전한 컬럼명 정제 함수 (공백/특수문자 -> _)
        def _clean_name(s: str) -> str:
            s = re.sub(r'\s+', '_', str(s))
            s = re.sub(r'[^0-9A-Za-z_]+', '_', s)
            s = re.sub(r'_+', '_', s).strip('_')
            if s and s[0].isdigit():
                s = f"feat_{s}"
            return s or "feature"

        # 4) 원-핫 생성 (학습 시점의 self.top_menus_에 한정)
        if 'menu' in Xe.columns:
            menu_str = Xe['menu'].astype(str)
            for m in self.top_menus_:
                Xe[f'menu_{_clean_name(m)}'] = (menu_str == str(m)).astype(int)

        # 5) 비수치 원본 컬럼 제거(문자열/날짜는 모델 입력에서 제외)
        for c in ('menu', 'store', 'date'):
            if c in Xe.columns:
                Xe.drop(columns=[c], inplace=True)

        # 6) 타입/결측 정리
        # 6-1) datetime -> 일수(int) 변환
        for c in Xe.columns:
            if np.issubdtype(Xe[c].dtype, np.datetime64):
                Xe[c] = Xe[c].view('int64') // 86_400_000_000_000  # 일 단위

        # 6-2) bool -> int
        bool_cols = Xe.select_dtypes(include=['bool']).columns
        if len(bool_cols):
            Xe[bool_cols] = Xe[bool_cols].astype(int)

        # 6-3) object/category -> 숫자 시도 (잔여 문자열은 NaN)
        obj_cols = Xe.select_dtypes(include=['object', 'string', 'category']).columns
        if len(obj_cols):
            Xe[obj_cols] = Xe[obj_cols].apply(pd.to_numeric, errors='coerce')

        # 6-4) 숫자 컬럼만 추출 + inf/NaN 처리
        num_cols = Xe.select_dtypes(include=[np.number]).columns
        Xe = Xe[num_cols].replace([np.inf, -np.inf], np.nan)
        Xe[num_cols] = Xe[num_cols].fillna(Xe[num_cols].median(numeric_only=True)).fillna(0)

        # 6-5) 상수(분산 0) 컬럼 제거 → 트리 분할이 의미 없는 피처 제거
        if len(num_cols):
            non_const = Xe.nunique(dropna=False) > 1
            Xe = Xe.loc[:, list(non_const.index[non_const])]

        # 7) 컬럼명 최종 정리(공백 제거) + 스키마 고정
        Xe.columns = Xe.columns.str.replace(r'\s+', '_', regex=True)

        if fit or getattr(self, 'feature_cols_', None) is None:
            # 학습 시 스키마 확정
            self.feature_cols_ = Xe.columns.tolist()
        else:
            # 예측 시: 학습 스키마에 맞춰 누락 컬럼 채우고 추가 컬럼은 버림
            missing = [c for c in self.feature_cols_ if c not in Xe.columns]
            for c in missing:
                Xe[c] = 0.0
            Xe = Xe[self.feature_cols_]

        # (선택) 디버그용 미니 체크
        # assert Xe.select_dtypes(exclude='number').empty, "Non-numeric columns remain!"

        return Xe


    # ---------- Fit / Predict ----------
    def fit(self, X: pd.DataFrame, y: np.ndarray):
        # 인코딩(스키마 고정)
        X_enc = self._encode_features(X, fit=True)

        # Zero-inflation 라벨(7일 중 하나라도 >0)
        has_sales = (y > 0).any(axis=1).astype(int)
        pos = has_sales.sum()
        neg = len(has_sales) - pos
        spw = (neg / max(pos, 1)) if pos > 0 else 1.0  # scale_pos_weight

        self.zero_classifier.set_params(scale_pos_weight=spw)
        self.zero_classifier.fit(X_enc, has_sales)

        # 일자별 회귀 (SMAPE가 0일 제외이므로, 0 타깃 가중치 줄이기 권장)
        self.daily_regressors = []
        for h in range(7):
            reg = XGBRegressor(
                n_estimators=300,
                max_depth=5,
                learning_rate=0.1,
                min_child_weight=1.0,
                subsample=0.8,
                colsample_bytree=0.8,
                reg_alpha=0.0,
                reg_lambda=1.0,
                random_state=42,
                n_jobs=-1
            )
            # 0 타깃은 가중치 낮게(평가 제외일 가능성 높음)
            w = np.where(y[:, h] <= 0, 0.3, 1.0)
            reg.fit(X_enc, y[:, h], sample_weight=w)
            self.daily_regressors.append(reg)

        # 핵심 메뉴 1일차 보조모델(블렌딩용) 
        for menu in self.core_menus: 
            mask = (X['menu'] == menu) 
            if mask.sum() > 10: 
                mreg = LGBMRegressor(n_estimators=200, learning_rate=0.05, random_state=42, n_jobs=-1) 
                mreg.fit(X_enc[mask], y[mask, 0]) 
                self.menu_models[menu] = mreg
        self._is_fitted = True

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        X_enc = self._encode_features(X, fit=False)
        has_sales_prob = self.zero_classifier.predict_proba(X_enc)[:, 1]

        preds = np.column_stack([reg.predict(X_enc) for reg in self.daily_regressors])

        # 핵심 메뉴 1일차 블렌딩(0.7*글로벌 + 0.3*메뉴전용)
        if self.menu_models:
            # 예측에서도 menu 원본이 필요
            menus = X['menu'].to_numpy()
            day0 = preds[:, 0].copy()
            for menu, mreg in self.menu_models.items():
                idx = (menus == menu)
                if idx.any():
                    day0[idx] = 0.7 * day0[idx] + 0.3 * mreg.predict(X_enc[idx])
            preds[:, 0] = day0


        # (중요) 확률 곱셈은 기본 비활성화 — SMAPE에 불리
        if self.use_zero_gate:
            zero_mask = (has_sales_prob < self.zero_gate_thr)
            preds[zero_mask] = 0.0

        preds = np.clip(preds, 0.0, None)  # 음수 방지
        return preds

# ----------------------------
# Mirasia class
# ----------------------------

class MirasiaPreprocessor:
    def __init__(self):
        # 미라시아 메뉴 특성 분류
        self.menu_categories = {
            'brunch_weekend': ['브런치(대인) 주말'],  # 주말 특화
            'brunch_weekday': ['브런치(대인) 주중'],  # 주중 특화  
            'brunch_package': ['미라시아 브런치 (패키지)', '브런치 2인 패키지'],  # 패키지
            'brunch_child': ['브런치(어린이)'],
            'group_brunch': ['(단체)브런치주중 36,000'],
            'beverages': ['애플망고 에이드', '버드와이저(무제한)', '스텔라(무제한)'],
            'others': ['BBQ Platter']
        }
    
    def categorize_menu(self, menu_name):
        """메뉴를 카테고리별로 분류"""
        for category, menus in self.menu_categories.items():
            if menu_name in menus:
                return category
        return 'others'
    
    def preprocess_mirasia_data(self, df):
        """미라시아 전용 전처리"""
        mirasia_df = df[df['store'] == '미라시아'].copy()
        
        # 1. 메뉴 카테고리 추가
        mirasia_df['menu_category'] = mirasia_df['menu'].apply(self.categorize_menu)
        
        # 2. 브런치 메뉴 플래그
        mirasia_df['is_brunch'] = mirasia_df['menu'].str.contains('브런치', na=False)
        
        # 3. 주말/주중 구분 강화 (브런치는 주말/주중 패턴이 명확)
        mirasia_df['is_weekend_brunch'] = (
            mirasia_df['is_brunch'] & mirasia_df['is_weekend']
        )
        
        # 4. 패키지 메뉴 플래그
        mirasia_df['is_package'] = mirasia_df['menu'].str.contains('패키지', na=False)
        
        # 5. 단체 메뉴 플래그
        mirasia_df['is_group_menu'] = mirasia_df['menu'].str.contains('단체', na=False)
        
        # 6. 음료 메뉴 플래그
        beverage_keywords = ['에이드', '무제한', '버드와이저', '스텔라']
        mirasia_df['is_beverage'] = mirasia_df['menu'].str.contains('|'.join(beverage_keywords), na=False)
        
        # 7. 주말/브런치 교호작용 피처
        mirasia_df['weekend_brunch_interaction'] = (
            mirasia_df['is_weekend'].astype(int) * mirasia_df['is_brunch'].astype(int)
        )
        
        # 8. 매출 임계값 기반 이진 분류 (52% 0매출 대응)
        mirasia_df['has_sales'] = (mirasia_df['sales'] > 0).astype(int)
        
        # 9. 브런치 시즌성 피처 (관광 성수기와 연관)
        mirasia_df = self._add_brunch_seasonality(mirasia_df)
        
        return mirasia_df
    
    def _add_brunch_seasonality(self, df):
        """브런치 메뉴의 계절성 피처 추가"""
        # 브런치 성수기 (봄, 여름, 연말연시)
        df['is_brunch_season'] = df['month'].isin([3, 4, 5, 6, 7, 8, 12, 1])
        
        # 브런치 메뉴의 계절별 가중치
        df['brunch_season_weight'] = 1.0
        df.loc[df['is_brunch'] & df['is_brunch_season'], 'brunch_season_weight'] = 1.3
        df.loc[df['is_brunch'] & ~df['is_brunch_season'], 'brunch_season_weight'] = 0.8
        
        return df
class MirasiaMLModel:
    def __init__(self):
        # 주중/주말 분리 모델
        self.weekday_model = LGBMRegressor(
            n_estimators=800,        # 1200 → 800 (과적합 방지)
            max_depth=5,            # 6 → 5 
            num_leaves=20,          # 31 → 20 (과적합 방지)
            learning_rate=0.05,     # 0.03 → 0.05 (더 안정적)
            feature_fraction=0.9,   # 0.8 → 0.9 (피처가 많지 않음)
            bagging_fraction=0.85,  # 0.8 → 0.85
            min_child_samples=15,   # 추가: 과적합 방지
            reg_alpha=0.1,          # 추가: L1 정규화
            reg_lambda=0.1,         # 추가: L2 정규화
            random_state=42
        )
        self._is_fitted = False
        self.weekend_model = LGBMRegressor(
            n_estimators=800,        # 1200 → 800 (과적합 방지)
            max_depth=5,            # 6 → 5 
            num_leaves=20,          # 31 → 20 (과적합 방지)
            learning_rate=0.05,     # 0.03 → 0.05 (더 안정적)
            feature_fraction=0.8,   # 0.8 → 0.9 (피처가 많지 않음)
            bagging_fraction=0.85,  # 0.8 → 0.85
            min_child_samples=15,   # 추가: 과적합 방지
            reg_alpha=0.1,          # 추가: L1 정규화
            reg_lambda=0.1,         # 추가: L2 정규화
            random_state=42
        )
        
        # 브런치 특화 모델
        self.brunch_model = XGBRegressor(
            n_estimators=600,       # 1000 → 600 (충분함)
            max_depth=5,           # 4 → 5 (브런치는 복잡한 패턴)
            learning_rate=0.05,    # 0.03 → 0.05
            min_child_weight=3,    # 추가: 과적합 방지
            subsample=0.85,        # 추가
            colsample_bytree=0.85, # 추가
            reg_alpha=0.05,        # 추가: 정규화
            random_state=42
        )
        self.zero_cls = XGBClassifier(
            n_estimators=300,      # 600 → 300 (충분함)
            max_depth=3,          # 4 → 3 (이진분류는 단순하게)
            learning_rate=0.1,    # 0.07 → 0.1
            min_child_weight=5,   # 추가: 불균형 데이터 대응
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42
        )

    def _count_consecutive_weekends(self, target_data):
        """연속 주말 개수 계산"""
        try:
            weekend_flags = target_data['is_weekend'].values
            max_consecutive = 0
            current_consecutive = 0
            
            for is_weekend in weekend_flags:
                if is_weekend:
                    current_consecutive += 1
                    max_consecutive = max(max_consecutive, current_consecutive)
                else:
                    current_consecutive = 0
            
            return max_consecutive
        except:
            return 0  # 오류시 기본값
    def create_mirasia_features(self, df, seq_length=SEQUENCELENGTH, mode='train'):
        """미라시아 특화 피처 생성"""
        features_list = []
        targets_list = []
        
        for menu in df['menu'].unique():
            menu_data = df[df['menu'] == menu].sort_values('date').reset_index(drop=True)
            
            # 모드별 조건 분기
            if mode == 'predict':
                if len(menu_data) < seq_length:  # 예측 모드: 28일만 필요
                    continue
            else:  # train 모드
                if len(menu_data) < seq_length + PREDICT:  # 훈련 모드: 35일 필요
                    continue
            
            if mode == 'predict':
                # 예측 모드: 마지막 28일로 1개 피처만 생성
                seq_data = menu_data.tail(seq_length)
                
                # 미래 7일 정보 생성
                last_date = seq_data['date'].iloc[-1]
                future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=7)
                future_weekends = future_dates.dayofweek.isin([5, 6]).sum()
                weekend_pattern = future_dates.dayofweek.isin([5, 6]).astype(int).tolist()
                future_month = future_dates[0].month
                future_start_dow = future_dates[0].dayofweek
                
                features = {
                    # 기본 통계
                    'sales_mean': seq_data['sales'].mean(),
                    'sales_std': seq_data['sales'].std(),
                    'sales_max': seq_data['sales'].max(),
                    'sales_last_7d_mean': seq_data['sales'].tail(7).mean(),
                    
                    # 주말/주중 분리 통계
                    'weekday_sales_mean': seq_data[~seq_data['is_weekend']]['sales'].mean(),
                    'weekend_sales_mean': seq_data[seq_data['is_weekend']]['sales'].mean(),
                    'weekend_ratio': seq_data[seq_data['is_weekend']]['sales'].mean() / (seq_data['sales'].mean() + 1e-8),
                    
                    # 브런치 특화
                    'is_brunch': 1 if '브런치' in menu else 0,
                    'is_weekend_brunch': 1 if ('브런치' in menu and '주말' in menu) else 0,
                    'is_weekday_brunch': 1 if ('브런치' in menu and '주중' in menu) else 0,
                    'is_package': 1 if '패키지' in menu else 0,
                    'is_child_menu': 1 if '어린이' in menu else 0,
                    'is_group_menu': 1 if '단체' in menu else 0,
                    'is_beverage': 1 if any(x in menu for x in ['에이드', '무제한', '버드와이저']) else 0,
                    
                    # 계절성 (브런치 성수기)
                    'is_brunch_season': future_month in [3,4,5,6,7,8,12,1],
                    
                    # 미래 주말 정보
                    'future_weekends': future_weekends,
                    'weekend_pattern': weekend_pattern,
                    
                    # 시간 정보
                    'month': future_month,
                    'start_day_of_week': future_start_dow,
                }
                
                features['menu'] = menu
                features_list.append(features)
                
            else:  # train 모드 - 기존 로직
                for i in range(len(menu_data) - seq_length - 7 + 1):
                    seq_data = menu_data.iloc[i:i+seq_length]
                    target_data = menu_data.iloc[i+seq_length:i+seq_length+7]
                    
                    # 기본 피처 + 미라시아 특화
                    features = {
                        # 기본 통계
                        'sales_mean': seq_data['sales'].mean(),
                        'sales_std': seq_data['sales'].std(),
                        'sales_max': seq_data['sales'].max(),
                        'sales_last_7d_mean': seq_data['sales'].tail(7).mean(),
                        
                        # 주말/주중 분리 통계
                        'weekday_sales_mean': seq_data[~seq_data['is_weekend']]['sales'].mean(),
                        'weekend_sales_mean': seq_data[seq_data['is_weekend']]['sales'].mean(),
                        'weekend_ratio': seq_data[seq_data['is_weekend']]['sales'].mean() / (seq_data['sales'].mean() + 1e-8),
                        
                        # 브런치 특화
                        'is_brunch': 1 if '브런치' in menu else 0,
                        'is_weekend_brunch': 1 if ('브런치' in menu and '주말' in menu) else 0,
                        'is_weekday_brunch': 1 if ('브런치' in menu and '주중' in menu) else 0,
                        'is_package': 1 if '패키지' in menu else 0,
                        'is_child_menu': 1 if '어린이' in menu else 0,
                        'is_group_menu': 1 if '단체' in menu else 0,
                        'is_beverage': 1 if any(x in menu for x in ['에이드', '무제한', '버드와이저']) else 0,
                        
                        # 계절성 (브런치 성수기)
                        'is_brunch_season': target_data['month'].iloc[0] in [3,4,5,6,7,8,12,1],
                        
                        # 미래 주말 정보
                        'future_weekends': target_data['is_weekend'].sum(),
                        'weekend_pattern': target_data['is_weekend'].astype(int).tolist(),
                        
                        # 시간 정보
                        'month': target_data['month'].iloc[0],
                        'start_day_of_week': target_data['day_of_week'].iloc[0],
                    }
                    
                    features['menu'] = menu
                    features_list.append(features)
                    targets_list.append(target_data['sales'].values)
        
        X = pd.DataFrame(features_list).fillna(0.0)
        y = np.array(targets_list) if targets_list else np.empty((0, 7))
        return X, y
    
    def fit(self, X, y):
        """모델 학습"""
        X_enc = self._encode_mirasia_features(X)
        
        # 브런치 메뉴 분리
        brunch_mask = X['is_brunch'] == 1
        non_brunch_mask = ~brunch_mask
        
        # 주중/주말 인덱스 벡터 만들기 (행별로 7길이 리스트)
        # X['weekend_pattern']은 create_mirasia_features에서 넣은 리스트형 컬럼
        weekend_pat = X['weekend_pattern'].apply(lambda v: np.asarray(v, dtype=int) if isinstance(v, list) else np.zeros(7, int)).values
        # stack: (n, 7)
        weekend_mat = np.vstack(weekend_pat)
        weekday_mat = 1 - weekend_mat

        # -------- 브런치 전용 모델 (7일 평균 or 주말가중 평균 등) --------
        if brunch_mask.sum() > 10:
            # 브런치는 주말이 중요하므로 '주말 평균'을 타깃으로 쓰는 편이 잘 맞음
            wcnt = weekend_mat[brunch_mask].sum(axis=1).clip(min=1)
            brunch_targets = (y[brunch_mask] * weekend_mat[brunch_mask]).sum(axis=1) / wcnt
            self.brunch_model.fit(X_enc[brunch_mask], np.log1p(brunch_targets))  # log1p 안정화
            self._use_log1p = True
        else:
            self._use_log1p = False
        
        # -------- 일반 메뉴: 주중 평균 / 주말 평균을 각각 학습 --------
        if non_brunch_mask.sum() > 10:
            # 주중 평균
            dcnt = weekday_mat[non_brunch_mask].sum(axis=1).clip(min=1)
            weekday_targets = (y[non_brunch_mask] * weekday_mat[non_brunch_mask]).sum(axis=1) / dcnt

            # 주말 평균
            wcnt = weekend_mat[non_brunch_mask].sum(axis=1).clip(min=1)
            weekend_targets = (y[non_brunch_mask] * weekend_mat[non_brunch_mask]).sum(axis=1) / wcnt

            # 0 타깃 가중치 낮추기(평가에서 0일 제외되는 경향 반영)
            ww = np.where(weekday_targets <= 0, 0.3, 1.0)
            ww_end = np.where(weekend_targets <= 0, 0.3, 1.0)

            self.weekday_model.fit(X_enc[non_brunch_mask], np.log1p(weekday_targets), sample_weight=ww)
            self.weekend_model.fit(X_enc[non_brunch_mask], np.log1p(weekend_targets), sample_weight=ww_end)
            self._use_log1p = True

        # fit() 말미
        has_sales = (y > 0).any(axis=1).astype(int)
        pos = has_sales.sum(); neg = len(has_sales) - pos
        spw = (neg / max(pos,1)) if pos>0 else 1.0
        self.zero_cls.set_params(scale_pos_weight=spw)
        self.zero_cls.fit(X_enc, has_sales)
        self._is_fitted = True
    
    def _encode_mirasia_features(self, X): 
        """미라시아 피처 인코딩""" 
        X_encoded = X.copy() 
        # 메뉴 카테고리 인코딩 
        brunch_menus = ['브런치(대인) 주말', '브런치(대인) 주중', '미라시아 브런치 (패키지)'] 
        for menu in brunch_menus: 
            X_encoded[f'menu_{menu}'] = (X['menu'] == menu).astype(int) 
            
        # 주말 패턴 인코딩 
        weekend_patterns = X['weekend_pattern'].apply(lambda x:sum(x) if isinstance(x, list) else 0) 
        X_encoded['weekend_days_count'] = weekend_patterns 
        # 불필요한 컬럼 제거 
        cols_to_drop = ['menu', 'weekend_pattern'] 
        X_encoded = X_encoded.drop([col for col in cols_to_drop if col in X_encoded.columns], axis=1) 

        return X_encoded
    def predict_eh(self, X):
        """개선된 예측 로직"""
        X_enc = self._encode_mirasia_features(X)
        n = len(X)
        preds = np.zeros((n, 7), dtype=float)
        
        # 기존 로직...
        
        # ✨ 개선점 1: 계절별 보정
        month_factors = {3: 1.1, 4: 1.15, 5: 1.2, 6: 1.25, 7: 1.3, 8: 1.25, 
                        9: 1.1, 10: 1.0, 11: 0.9, 12: 1.0, 1: 0.95, 2: 0.9}
        
        for i in range(n):
            month = X.iloc[i]['month']
            if X.iloc[i]['is_brunch']:
                preds[i] *= month_factors.get(month, 1.0)
        
        # ✨ 개선점 2: 스무딩 (급격한 변화 방지)
        for i in range(n):
            for day in range(1, 7):
                if preds[i, day] > preds[i, day-1] * 2.5:  # 250% 이상 증가 제한
                    preds[i, day] = preds[i, day-1] * 2.5
                elif preds[i, day] < preds[i, day-1] * 0.4:  # 60% 이상 감소 제한
                    preds[i, day] = preds[i, day-1] * 0.4
        
        # ✨ 개선점 3: 적응적 임계값
        prob = self.zero_cls.predict_proba(X_enc)[:,1]
        
        # 브런치는 더 관대한 임계값
        brunch_mask = (X['is_brunch'] == 1).values
        threshold = np.where(brunch_mask, 0.1, 0.15)  # 브런치 0.1, 일반 0.15
        
        mask = prob < threshold
        preds[mask] = 0.0
        
        return preds
    def predict(self, X):
        X_enc = self._encode_mirasia_features(X)
        n = len(X)
        preds = np.zeros((n, 7), dtype=float)

        brunch_mask = (X['is_brunch'] == 1).values
        non_brunch_mask = ~brunch_mask

        weekend_pat = X['weekend_pattern'].apply(lambda v: np.asarray(v, dtype=int) if isinstance(v, list) else np.zeros(7, int)).values
        weekend_mat = np.vstack(weekend_pat)
        weekday_mat = 1 - weekend_mat

        # --- 브런치 ---
        if brunch_mask.any():
            bp = self.brunch_model.predict(X_enc[brunch_mask])
            if getattr(self, '_use_log1p', False):
                bp = np.expm1(bp)
            # 주말 가중치 (간단 버전 유지)
            weekend_days = weekend_mat[brunch_mask].sum(axis=1)
            w = np.where(weekend_days > 3, 1.2, 0.8)
            bp = (bp * w)
            # 주중/주말에 동일값을 깔아도 되지만, 주말엔 +10% 같은 미세 보정 가능
            preds[brunch_mask] = (bp[:, None] * (0.9 * weekday_mat[brunch_mask] + 1.1 * weekend_mat[brunch_mask]))

        # --- 일반 메뉴 ---
        if non_brunch_mask.any():
            wk = self.weekday_model.predict(X_enc[non_brunch_mask])
            we = self.weekend_model.predict(X_enc[non_brunch_mask])
            if getattr(self, '_use_log1p', False):
                wk = np.expm1(wk); we = np.expm1(we)
            # 요일별로 선택
            preds[non_brunch_mask] = (wk[:, None] * weekday_mat[non_brunch_mask] +
                                    we[:, None] * weekend_mat[non_brunch_mask])

        # 음수 방지 + 소폭 스무딩(선택)
        preds = np.clip(preds, 0.0, None)
        # predict() 말미
        prob = self.zero_cls.predict_proba(X_enc)[:,1]
        preds[prob < 0.15] = 0.0   # 임계 0.10~0.30 검증으로 튜닝
        
        return preds


# ----------------------------
# Main restraunts 
# ----------------------------

class MainRestaurantModel:
    def __init__(self):
        # 업장별 특성 정의
        self.store_config = {
            '포레스트릿': {'scale': 'large', 'avg_sales': 47.84, 'volatility': 'low'},
            '화담숲주막': {'scale': 'large', 'avg_sales': 34.38, 'volatility': 'medium'},  
            '화담숲카페': {'scale': 'medium', 'avg_sales': 23.55, 'volatility': 'medium'},
            '카페테리아': {'scale': 'medium', 'avg_sales': 18.86, 'volatility': 'low'}
        }
        
        # 업장별 스케일 대응 모델
        self.large_scale_model = XGBRegressor(
            n_estimators=300,
            max_depth=8,
            learning_rate=0.08,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42
        )
        
        self.medium_scale_model = LGBMRegressor(
            n_estimators=250,
            max_depth=6,
            learning_rate=0.1,
            feature_fraction=0.8,
            bagging_fraction=0.8,
            random_state=42
        )
        
        # 멀티타겟 회귀를 위한 개별 일자별 모델
        self.daily_models = {}
        
        # 업장별 개별 모델 (필요시)
        self.store_specific_models = {}
        
        # 피처 스케일러
        self.scalers = {}
        self.label_encoders = {}
        self._is_fitted = False

    def create_main_restaurant_features(self, df, seq_length=SEQUENCELENGTH, mode='train'):
        """메인 레스토랑 특화 피처 생성"""
        features_list = []
        targets_list = []
        store_info_list = []
        
        main_stores = ['포레스트릿', '카페테리아', '화담숲주막', '화담숲카페']
        
        for store in main_stores:
            store_data = df[df['store'] == store]
            
            for menu in store_data['menu'].unique():
                menu_data = store_data[store_data['menu'] == menu].sort_values('date').reset_index(drop=True)
                
                # 모드별 조건 분기
                if mode == 'predict':
                    if len(menu_data) < seq_length:  # 예측 모드: 28일만 필요
                        continue
                else:  # train 모드
                    if len(menu_data) < seq_length + PREDICT:  # 훈련 모드: 35일 필요
                        continue
                
                if mode == 'predict':
                    # 예측 모드: 마지막 28일로 1개 피처만 생성
                    seq_data = menu_data.tail(seq_length)
                    
                    # 미래 7일 정보 생성
                    last_date = seq_data['date'].iloc[-1]
                    future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=7)
                    
                    # 가상의 target_data 생성 (피처 생성용)
                    target_data = pd.DataFrame({
                        'date': future_dates,
                        'month': future_dates.month,
                        'day_of_week': future_dates.dayofweek,
                        'is_weekend': future_dates.dayofweek.isin([5, 6]),
                        'is_holiday_season': future_dates.month.isin([7, 8, 12, 1])
                    })
                    
                    # 고급 통계 피처 생성
                    features = self._create_advanced_features(seq_data, target_data, store, menu)
                    
                    features_list.append(features)
                    store_info_list.append({'store': store, 'menu': menu})
                    
                else:  # train 모드 - 기존 로직
                    for i in range(len(menu_data) - seq_length - PREDICT + 1):
                        seq_data = menu_data.iloc[i:i+seq_length]
                        target_data = menu_data.iloc[i+seq_length:i+seq_length+7]
                        
                        # 고급 통계 피처 생성
                        features = self._create_advanced_features(seq_data, target_data, store, menu)
                        
                        features_list.append(features)
                        targets_list.append(target_data['sales'].values)
                        store_info_list.append({'store': store, 'menu': menu})
        
        X = pd.DataFrame(features_list).fillna(0.0)
        y = np.array(targets_list) if targets_list else np.empty((0, 7))
        return X, y, store_info_list

    def _create_advanced_features(self, seq_data, target_data, store, menu):
        """고급 피처 생성 (train/predict 모드 모두 지원)"""
        
        # 기본 통계 피처
        sales = seq_data['sales'].values
        features = {
            # 중심 경향성
            'sales_mean': np.mean(sales),
            'sales_median': np.median(sales),
            'sales_std': np.std(sales),
            'sales_cv': np.std(sales) / (np.mean(sales) + 1e-8),  # 변동계수
            
            # 분포 특성
            'sales_skew': self._calculate_skewness(sales),
            'sales_kurtosis': self._calculate_kurtosis(sales),
            'sales_min': np.min(sales),
            'sales_max': np.max(sales),
            'sales_range': np.max(sales) - np.min(sales),
            'sales_iqr': np.percentile(sales, 75) - np.percentile(sales, 25),
            
            # 시간 구간별 통계
            'sales_first_week': np.mean(sales[:7]),
            'sales_second_week': np.mean(sales[7:14]),
            'sales_third_week': np.mean(sales[14:21]),
            'sales_last_week': np.mean(sales[21:28]),
            
            # 최근 패턴
            'sales_last_7d_mean': np.mean(sales[-7:]),
            'sales_last_3d_mean': np.mean(sales[-3:]),
            'sales_last_1d': sales[-1],
            
            # 트렌드 분석
            'sales_trend': self._calculate_trend(sales),
            'sales_acceleration': self._calculate_acceleration(sales),
            'recent_vs_early_ratio': np.mean(sales[-7:]) / (np.mean(sales[:7]) + 1e-8),
            
            # 요일별 패턴
            'monday_avg': seq_data[seq_data['day_of_week'] == 0]['sales'].mean(),
            'tuesday_avg': seq_data[seq_data['day_of_week'] == 1]['sales'].mean(),
            'wednesday_avg': seq_data[seq_data['day_of_week'] == 2]['sales'].mean(),
            'thursday_avg': seq_data[seq_data['day_of_week'] == 3]['sales'].mean(),
            'friday_avg': seq_data[seq_data['day_of_week'] == 4]['sales'].mean(),
            'saturday_avg': seq_data[seq_data['day_of_week'] == 5]['sales'].mean(),
            'sunday_avg': seq_data[seq_data['day_of_week'] == 6]['sales'].mean(),
            
            # 주말/주중 패턴
            'weekday_mean': seq_data[~seq_data['is_weekend']]['sales'].mean(),
            'weekend_mean': seq_data[seq_data['is_weekend']]['sales'].mean(),
            'weekend_boost_ratio': seq_data[seq_data['is_weekend']]['sales'].mean() / 
                                (seq_data[~seq_data['is_weekend']]['sales'].mean() + 1e-8),
            
            # 변동성 지표
            'sales_volatility_7d': np.std(sales[-7:]),
            'sales_volatility_14d': np.std(sales[-14:]),
            'max_consecutive_increase': self._max_consecutive_pattern(sales, 'increase'),
            'max_consecutive_decrease': self._max_consecutive_pattern(sales, 'decrease'),
            
            # 0매출 관련
            'zero_sales_count': np.sum(sales == 0),
            'zero_sales_ratio': np.mean(sales == 0),
            'days_since_last_zero': self._days_since_last_zero(sales),
            'longest_zero_streak': self._longest_zero_streak(sales),
            
            # 업장 특성 피처
            'store_scale': self.store_config[store]['scale'],
            'store_avg_baseline': self.store_config[store]['avg_sales'],
            'relative_performance': np.mean(sales) / self.store_config[store]['avg_sales'],
            
            # 메뉴 특성
            'menu_length': len(menu),
            'is_main_dish': 1 if any(x in menu.lower() for x in ['불고기', '갈비', '스테이크']) else 0,
            'is_beverage': 1 if any(x in menu.lower() for x in ['커피', '차', '음료', '주스']) else 0,
            'is_dessert': 1 if any(x in menu.lower() for x in ['케이크', '디저트', '아이스크림']) else 0,
            'is_set_menu': 1 if '세트' in menu or '정식' in menu else 0,
            
            # 시즌 정보 (안전하게 접근)
            'target_month': int(target_data['month'].iloc[0]) if len(target_data) > 0 else 1,
            'is_peak_season': int(target_data['is_holiday_season'].iloc[0]) if len(target_data) > 0 else 0,
            'target_start_dow': int(target_data['day_of_week'].iloc[0]) if len(target_data) > 0 else 0,
            
            # 미래 요일 분포
            'future_weekends': int(target_data['is_weekend'].sum()) if len(target_data) > 0 else 0,
            'future_fridays': int((target_data['day_of_week'] == 4).sum()) if len(target_data) > 0 else 0,
            'future_saturdays': int((target_data['day_of_week'] == 5).sum()) if len(target_data) > 0 else 0,
            'future_sundays': int((target_data['day_of_week'] == 6).sum()) if len(target_data) > 0 else 0,
            
            # 라그 피처 (과거 동일 요일)
            'same_dow_1w_ago': sales[-7] if len(sales) >= 7 else 0,
            'same_dow_2w_ago': sales[-14] if len(sales) >= 14 else 0,
            'same_dow_3w_ago': sales[-21] if len(sales) >= 21 else 0,
            'same_dow_4w_ago': sales[-28] if len(sales) >= 28 else 0,
        }
        
        # 카테고리 변수
        features['store'] = store
        features['menu'] = menu
        
        return features
    def _calculate_skewness(self, data):
        """왜도 계산"""
        if len(data) < 3:
            return 0
        mean = np.mean(data)
        std = np.std(data)
        if std == 0:
            return 0
        return np.mean(((data - mean) / std) ** 3)
    
    def _calculate_kurtosis(self, data):
        """첨도 계산"""
        if len(data) < 4:
            return 0
        mean = np.mean(data)
        std = np.std(data)
        if std == 0:
            return 0
        return np.mean(((data - mean) / std) ** 4) - 3
    
    def _calculate_trend(self, sales):
        """선형 트렌드 기울기"""
        if len(sales) < 2:
            return 0
        x = np.arange(len(sales))
        return np.polyfit(x, sales, 1)[0]
    
    def _calculate_acceleration(self, sales):
        """가속도 (2차 미분)"""
        if len(sales) < 3:
            return 0
        x = np.arange(len(sales))
        return np.polyfit(x, sales, 2)[0] * 2
    
    def _max_consecutive_pattern(self, sales, pattern='increase'):
        """최대 연속 증가/감소 일수"""
        if len(sales) < 2:
            return 0
        
        diffs = np.diff(sales)
        if pattern == 'increase':
            condition = diffs > 0
        else:
            condition = diffs < 0
            
        max_consecutive = 0
        current_consecutive = 0
        
        for is_pattern in condition:
            if is_pattern:
                current_consecutive += 1
                max_consecutive = max(max_consecutive, current_consecutive)
            else:
                current_consecutive = 0
                
        return max_consecutive
    
    def _days_since_last_zero(self, sales):
        """마지막 0매출 이후 경과일"""
        zero_indices = np.where(sales == 0)[0]
        if len(zero_indices) == 0:
            return len(sales)
        return len(sales) - 1 - zero_indices[-1]
    
    def _longest_zero_streak(self, sales):
        """최장 연속 0매출 일수"""
        max_streak = 0
        current_streak = 0
        
        for sale in sales:
            if sale == 0:
                current_streak += 1
                max_streak = max(max_streak, current_streak)
            else:
                current_streak = 0
                
        return max_streak
    
    def fit(self, X, y, store_info):
        """모델 학습"""
        # 피처 전처리
        X_processed = self._preprocess_features(X)
        
        # 업장별 스케일 분리
        large_scale_mask = X['store_scale'] == 'large'
        medium_scale_mask = X['store_scale'] == 'medium'
        
        # 대형 업장 모델 학습 (포레스트릿, 화담숲주막)
        if large_scale_mask.sum() > 0:
            # 멀티아웃풋으로 7일 동시 예측
            large_multi_model = MultiOutputRegressor(
                XGBRegressor(n_estimators=300, max_depth=8, learning_rate=0.08, random_state=42)
            )
            large_multi_model.fit(X_processed[large_scale_mask], y[large_scale_mask])
            self.large_scale_model = large_multi_model
        
        # 중형 업장 모델 학습 (카페테리아, 화담숲카페)
        if medium_scale_mask.sum() > 0:
            medium_multi_model = MultiOutputRegressor(
                LGBMRegressor(n_estimators=250, max_depth=6, learning_rate=0.1, random_state=42)
            )
            medium_multi_model.fit(X_processed[medium_scale_mask], y[medium_scale_mask])
            self.medium_scale_model = medium_multi_model
        
        # 개별 일자별 모델 (정밀도 향상)
        for day in range(7):
            day_model = XGBRegressor(
                n_estimators=200,
                max_depth=6,
                learning_rate=0.1,
                random_state=42
            )
            day_model.fit(X_processed, y[:, day])
            self.daily_models[f'day_{day}'] = day_model
        
        # 업장별 특화 모델 (필요시)
        for store in ['포레스트릿', '화담숲주막']:  # 고매출 업장만
            store_mask = X['store'] == store
            if store_mask.sum() > 20:  # 충분한 데이터가 있는 경우
                store_model = MultiOutputRegressor(
                    XGBRegressor(n_estimators=150, max_depth=7, learning_rate=0.12, random_state=42)
                )
                store_model.fit(X_processed[store_mask], y[store_mask])
                self.store_specific_models[store] = store_model
        self._is_fitted = True
        
    def _preprocess_features(self, X):
        """피처 전처리"""
        X_processed = X.copy()
        
        # 카테고리 변수 인코딩
        if 'store' not in self.label_encoders:
            self.label_encoders['store'] = LabelEncoder()
            X_processed['store_encoded'] = self.label_encoders['store'].fit_transform(X['store'])
        else:
            X_processed['store_encoded'] = self.label_encoders['store'].transform(X['store'])
        
        if 'menu' not in self.label_encoders:
            # 메뉴는 해시 인코딩 (너무 많은 카테고리)
            X_processed['menu_hash'] = X['menu'].apply(hash).apply(lambda x: abs(x) % 1000)
        
        # 스케일 인코딩
        scale_map = {'large': 1, 'medium': 0}
        X_processed['store_scale_encoded'] = X['store_scale'].map(scale_map)
        
        # 숫자형 피처만 선택
        numeric_features = X_processed.select_dtypes(include=[np.number]).columns
        X_numeric = X_processed[numeric_features]
        
        # NaN 처리
        X_numeric = X_numeric.fillna(0)
        
        return X_numeric
    
    def predict(self, X):
        """예측"""
        X_processed = self._preprocess_features(X)
        
        # 업장별 스케일 분리 예측
        large_scale_mask = X['store_scale'] == 'large'
        medium_scale_mask = X['store_scale'] == 'medium'
        
        predictions = np.zeros((len(X), 7))
        
        # 대형 업장 예측
        if large_scale_mask.sum() > 0:
            large_pred = self.large_scale_model.predict(X_processed[large_scale_mask])
            predictions[large_scale_mask] = large_pred
        
        # 중형 업장 예측
        if medium_scale_mask.sum() > 0:
            medium_pred = self.medium_scale_model.predict(X_processed[medium_scale_mask])
            predictions[medium_scale_mask] = medium_pred
        
        # 개별 업장 모델 보정 (고매출 업장)
        for store in self.store_specific_models:
            store_mask = X['store'] == store
            if store_mask.sum() > 0:
                store_pred = self.store_specific_models[store].predict(X_processed[store_mask])
                # 가중 평균 (기본 모델 0.7 + 특화 모델 0.3)
                predictions[store_mask] = 0.7 * predictions[store_mask] + 0.3 * store_pred
        
        return predictions
    
    def get_feature_importance(self):
        """피처 중요도 반환"""
        importance_dict = {}
        
        # 대형 업장 모델 중요도
        if hasattr(self.large_scale_model, 'estimators_'):
            large_importance = np.mean([
                estimator.feature_importances_ 
                for estimator in self.large_scale_model.estimators_
            ], axis=0)
            importance_dict['large_scale'] = large_importance
        
        return importance_dict
    # 모델 사용 예시
def train_main_restaurant_model(train_df):
    # 모델 초기화
    main_model = MainRestaurantModel()
    
    # 피처 생성
    X, y, store_info = main_model.create_main_restaurant_features(train_df)
    
    # 모델 학습
    main_model.fit(X, y, store_info)
    
    # 피처 중요도 확인
    importance = main_model.get_feature_importance()
    print("주요 피처 중요도:", importance)
    
    return main_model

# 예측
def predict_main_restaurants(model, test_data):
    X_test, _, _ = model.create_main_restaurant_features(test_data)
    predictions = model.predict(X_test)
    return predictions
# ----------------------------
# reagular restraunt , special restraunts / baseline model
# ----------------------------

class RegularOutletModel:
    def __init__(self):
        # 일반 업장 특성: 느티나무 셀프BBQ (평균 5.70, 0매출 60.1%)
        self.outlet_config = {
            'store_name': '느티나무 셀프BBQ',
            'avg_sales': 5.70,
            'zero_ratio': 0.601,
            'service_type': 'self_service',
            'menu_count': 23
        }
        
        # Zero-inflation 이중 모델
        self.zero_classifier = XGBClassifier(
            n_estimators=100,
            max_depth=4,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=2.0,  # 불균형 데이터 대응
            random_state=42,
            verbosity=0
        )
        
        self.sales_regressor = LGBMRegressor(
            n_estimators=150,
            max_depth=5,
            learning_rate=0.1,
            feature_fraction=0.8,
            bagging_fraction=0.8,
            min_child_samples=10,
            random_state=42,
            verbosity=-1
        )
        
        # 일자별 개별 모델 (7일)
        self.daily_models = {}
        
        # 학습 상태 추적
        self._is_fitted = False
        
    def create_regular_outlet_features(self, df, seq_length=28, mode='train'):
        """일반 업장(셀프BBQ) 특화 피처 생성"""
        print(f"   📊 일반 업장 피처 생성:")
        print(f"      - 입력 데이터: {len(df)}행")
        
        features_list = []
        targets_list = []
        
        outlet_data = df[df['store'] == '느티나무 셀프BBQ']
        print(f"      - 느티나무 셀프BBQ 데이터: {len(outlet_data)}행")
        
        if len(outlet_data) == 0:
            print("      ❌ 느티나무 셀프BBQ 데이터 없음")
            return pd.DataFrame(), np.array([]).reshape(0, 7)
        
        valid_sequences = 0
        
        for menu in outlet_data['menu'].unique():
            menu_data = outlet_data[outlet_data['menu'] == menu].sort_values('date').reset_index(drop=True)
            
            # 모드별 조건 분기
            if mode == 'predict':
                if len(menu_data) < seq_length:  # 예측 모드: 28일만 필요
                    continue
            else:  # train 모드
                if len(menu_data) < seq_length + 7:  # 훈련 모드: 35일 필요
                    continue
            
            if mode == 'predict':
                # 예측 모드: 마지막 28일로 1개 피처만 생성
                try:
                    seq_data = menu_data.tail(seq_length)
                    
                    # 미래 7일 정보 생성
                    last_date = seq_data['date'].iloc[-1]
                    future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=7)
                    
                    # 가상의 target_data 생성
                    target_data = pd.DataFrame({
                        'date': future_dates,
                        'month': future_dates.month,
                        'day_of_week': future_dates.dayofweek,
                        'is_weekend': future_dates.dayofweek.isin([5, 6])
                    })
                    
                    # 셀프서비스 특화 피처 생성
                    features = self._create_self_service_features(seq_data, target_data, menu)
                    features_list.append(features)
                    valid_sequences += 1
                    
                except Exception as e:
                    continue
                    
            else:  # train 모드 - 기존 로직
                for i in range(len(menu_data) - seq_length - 7 + 1):
                    try:
                        seq_data = menu_data.iloc[i:i+seq_length]
                        target_data = menu_data.iloc[i+seq_length:i+seq_length+7]
                        
                        # 셀프서비스 특화 피처 생성
                        features = self._create_self_service_features(seq_data, target_data, menu)
                        features_list.append(features)
                        targets_list.append(target_data['sales'].values)
                        valid_sequences += 1
                        
                    except Exception as e:
                        continue
        
        print(f"      ✅ {valid_sequences}개 시퀀스 생성")
        
        if len(features_list) == 0:
            return pd.DataFrame(), np.array([]).reshape(0, 7)
        
        X = pd.DataFrame(features_list).fillna(0.0)
        y = np.array(targets_list) if targets_list else np.empty((0, 7))
        return X, y

    
    def _create_self_service_features(self, seq_data, target_data, menu):
        """셀프서비스 업장 특화 피처"""
        sales = seq_data['sales'].values
        
        features = {
            # 기본 통계 (0매출 고려)
            'sales_mean': float(seq_data['sales'].mean()),
            'sales_std': float(seq_data['sales'].std()) if len(sales) > 1 else 0.0,
            'sales_max': float(seq_data['sales'].max()),
            'sales_median': float(seq_data['sales'].median()),
            'non_zero_mean': float(seq_data[seq_data['sales'] > 0]['sales'].mean()) if (seq_data['sales'] > 0).any() else 0.0,
            'non_zero_count': int((seq_data['sales'] > 0).sum()),
            'non_zero_ratio': float((seq_data['sales'] > 0).mean()),
            
            # Zero-inflation 특화 피처
            'zero_count': int((seq_data['sales'] == 0).sum()),
            'zero_ratio': float((seq_data['sales'] == 0).mean()),
            'consecutive_zeros': int(self._max_consecutive_zeros(sales)),
            'days_since_last_sale': int(self._days_since_last_sale(sales)),
            'longest_active_period': int(self._longest_active_period(sales)),
            
            # 셀프서비스 패턴 (추정)
            'weekday_activity': float(seq_data[~seq_data['is_weekend']]['sales'].mean()) if (~seq_data['is_weekend']).any() else 0.0,
            'weekend_activity': float(seq_data[seq_data['is_weekend']]['sales'].mean()) if seq_data['is_weekend'].any() else 0.0,
            'midweek_depression': float(seq_data[seq_data['day_of_week'].isin([1, 2, 3])]['sales'].mean()) if seq_data['day_of_week'].isin([1, 2, 3]).any() else 0.0,
            'weekend_boost_ratio': float(seq_data[seq_data['is_weekend']]['sales'].mean() / (seq_data[~seq_data['is_weekend']]['sales'].mean() + 1e-8)),
            
            # 메뉴 특성 (셀프BBQ 특화)
            'is_main_meat': 1 if any(x in str(menu).lower() for x in ['불고기', '갈비', '삼겹살', '목살']) else 0,
            'is_side_dish': 1 if any(x in str(menu).lower() for x in ['밑반찬', '김치', '쌈', '야채']) else 0,
            'is_rice_dish': 1 if any(x in str(menu).lower() for x in ['밥', '비빔밥', '덮밥']) else 0,
            'is_beverage': 1 if any(x in str(menu).lower() for x in ['음료', '맥주', '소주', '물']) else 0,
            'is_rental': 1 if any(x in str(menu).lower() for x in ['대여', '그늘집', '의자']) else 0,
            'is_utensil': 1 if any(x in str(menu).lower() for x in ['수저', '접시', '컵']) else 0,
            
            # 시간 정보
            'target_month': int(target_data['month'].iloc[0]),
            'target_weekends': int(target_data['is_weekend'].sum()),
            'target_start_dow': int(target_data['day_of_week'].iloc[0]),
            'is_peak_season': 1 if target_data['month'].iloc[0] in [6, 7, 8] else 0,  # 여름
            
            # 변동성
            'sales_volatility': float(seq_data['sales'].std() / (seq_data['sales'].mean() + 1e-8)),
            'recent_trend': float((seq_data['sales'].tail(7).mean() - seq_data['sales'].head(7).mean()) if len(seq_data) >= 14 else 0),
            
            # 라그 피처
            'lag_1w': float(sales[-7]) if len(sales) >= 7 else 0.0,
            'lag_2w': float(sales[-14]) if len(sales) >= 14 else 0.0,
        }
        
        # NaN 처리
        for key, value in features.items():
            if pd.isna(value) or np.isinf(value):
                features[key] = 0.0
        
        features['menu'] = str(menu)
        return features
    
    def _max_consecutive_zeros(self, sales):
        """최대 연속 0매출 일수"""
        max_zeros = 0
        current_zeros = 0
        for sale in sales:
            if sale == 0:
                current_zeros += 1
                max_zeros = max(max_zeros, current_zeros)
            else:
                current_zeros = 0
        return max_zeros
    
    def _days_since_last_sale(self, sales):
        """마지막 매출 이후 경과일"""
        non_zero_indices = np.where(sales > 0)[0]
        if len(non_zero_indices) == 0:
            return len(sales)
        return len(sales) - 1 - non_zero_indices[-1]
    
    def _longest_active_period(self, sales):
        """최장 연속 매출 기간"""
        max_active = 0
        current_active = 0
        for sale in sales:
            if sale > 0:
                current_active += 1
                max_active = max(max_active, current_active)
            else:
                current_active = 0
        return max_active
    
    def fit(self, X, y):
        """✨ 누락된 fit 메서드 구현 ✨"""
        print(f"   📚 일반 업장 모델 학습:")
        print(f"      - 피처 수: {len(X)} x {len(X.columns) if len(X) > 0 else 0}")
        print(f"      - 타겟 수: {len(y)} x {y.shape[1] if len(y) > 0 else 0}")
        
        if len(X) == 0 or len(y) == 0:
            print("      ❌ 학습 데이터 없음")
            self._is_fitted = False
            return
        
        try:
            # 피처 전처리
            X_processed = self._preprocess_features(X)
            
            # Zero-inflation 이진 분류기 학습
            has_sales = (y > 0).any(axis=1)  # 7일 중 하나라도 매출이 있으면 1
            print(f"      - 매출 있는 비율: {has_sales.mean():.1%}")
            
            if has_sales.sum() > 5:  # 최소 5개 양성 샘플
                # 클래스 불균형 조정
                pos_weight = (len(has_sales) - has_sales.sum()) / max(has_sales.sum(), 1)
                self.zero_classifier.set_params(scale_pos_weight=pos_weight)
                self.zero_classifier.fit(X_processed, has_sales)
                print("      ✅ Zero classifier 학습 완료")
            else:
                print("      ⚠️ 양성 샘플 부족 - Zero classifier 스킵")
            
            # 매출이 있는 경우만으로 회귀 모델 학습
            positive_mask = has_sales
            if positive_mask.sum() > 5:
                # 전체 7일 평균을 타겟으로 사용
                avg_targets = y[positive_mask].mean(axis=1)
                
                # 가중치 (0에 가까운 값은 낮은 가중치)
                weights = np.where(avg_targets <= 1, 0.5, 1.0)
                
                self.sales_regressor.fit(
                    X_processed[positive_mask], 
                    avg_targets,
                    sample_weight=weights
                )
                print("      ✅ Sales regressor 학습 완료")
                
                # 일자별 개별 모델 (선택적)
                if len(X_processed[positive_mask]) > 20:
                    for day in range(7):
                        day_model = LGBMRegressor(
                            n_estimators=50,
                            max_depth=4,
                            learning_rate=0.15,
                            random_state=42,
                            verbosity=-1
                        )
                        day_targets = y[positive_mask, day]
                        day_weights = np.where(day_targets <= 1, 0.5, 1.0)
                        
                        day_model.fit(X_processed[positive_mask], day_targets, sample_weight=day_weights)
                        self.daily_models[f'day_{day}'] = day_model
                    
                    print(f"      ✅ 일자별 모델 {len(self.daily_models)}개 학습 완료")
            
            self._is_fitted = True
            print("      ✅ 전체 학습 완료")
            
        except Exception as e:
            print(f"      ❌ 학습 오류: {e}")
            self._is_fitted = False
    
    def _preprocess_features(self, X):
        """피처 전처리"""
        X_processed = X.copy()
        
        # 메뉴 인코딩
        if 'menu' in X_processed.columns:
            X_processed['menu_hash'] = X_processed['menu'].apply(lambda x: hash(str(x)) % 100)
            X_processed = X_processed.drop('menu', axis=1)
        
        # 숫자형만 선택
        numeric_cols = X_processed.select_dtypes(include=[np.number]).columns
        X_processed = X_processed[numeric_cols]
        
        # NaN, inf 처리
        X_processed = X_processed.fillna(0)
        X_processed = X_processed.replace([np.inf, -np.inf], 0)
        
        return X_processed
    
    def predict(self, X):
        """✨ 누락된 predict 메서드 구현 ✨"""
        if not getattr(self, '_is_fitted', False):
            print("      ⚠️ 모델이 학습되지 않음 - 기본값 반환")
            return np.full((len(X), 7), 5.0)
        
        try:
            X_processed = self._preprocess_features(X)
            predictions = np.zeros((len(X), 7))
            
            # 매출 여부 예측
            try:
                has_sales_prob = self.zero_classifier.predict_proba(X_processed)[:, 1]
            except:
                has_sales_prob = np.full(len(X), 0.4)  # 기본값
            
            # 매출량 예측
            try:
                avg_sales = self.sales_regressor.predict(X_processed)
            except:
                avg_sales = np.full(len(X), self.outlet_config['avg_sales'])
            
            # 일자별 예측 또는 평균값 분배
            if self.daily_models:
                # 일자별 개별 예측
                for day in range(7):
                    if f'day_{day}' in self.daily_models:
                        try:
                            day_pred = self.daily_models[f'day_{day}'].predict(X_processed)
                            predictions[:, day] = day_pred
                        except:
                            predictions[:, day] = avg_sales
                    else:
                        predictions[:, day] = avg_sales
            else:
                # 평균값을 7일에 분배 (주말 가중치 적용)
                for i in range(len(X)):
                    base_pred = avg_sales[i]
                    
                    # 간단한 요일 패턴 (주말에 약간 높게)
                    for day in range(7):
                        if day in [5, 6]:  # 토, 일
                            predictions[i, day] = base_pred * 1.1
                        else:
                            predictions[i, day] = base_pred * 0.95
            
            # Zero-inflation 적용
            threshold = 0.2  # 20% 이하면 0으로
            zero_mask = has_sales_prob < threshold
            predictions[zero_mask] = 0
            
            # 음수 방지
            predictions = np.clip(predictions, 0, None)
            
            return predictions
            
        except Exception as e:
            print(f"      ❌ 예측 오류: {e}")
            return np.full((len(X), 7), 3.0)
class SpecialVenueModel:
    def __init__(self):
        # 특수 업장 특성
        self.venue_config = {
            '연회장': {'avg_sales': 2.32, 'zero_ratio': 0.661, 'type': 'event_based'},
            '라그로타': {'avg_sales': 1.31, 'zero_ratio': 0.598, 'type': 'specialty_dining'}
        }
        
        # 이벤트 기반 예측 모델
        self.event_classifier = XGBClassifier(
            n_estimators=80,
            max_depth=3,
            learning_rate=0.15,
            scale_pos_weight=5.0,  # 높은 불균형 대응
            random_state=42,
            verbosity=0
        )
        
        # 조건부 회귀 모델
        self.conditional_regressor = LGBMRegressor(
            n_estimators=100,
            max_depth=4,
            learning_rate=0.15,
            min_child_samples=3,
            feature_fraction=0.7,
            bagging_fraction=0.7,
            random_state=42,
            verbosity=-1
        )
        
        # 업장별 개별 모델
        self.venue_specific_models = {}
        
        # 학습 상태
        self._is_fitted = False
        
    def create_special_venue_features(self, df, seq_length=28, mode='train'):
        """특수 업장 특화 피처 생성"""
        print(f"   🎭 특수 업장 피처 생성:")
        
        features_list = []
        targets_list = []
        venue_info_list = []
        
        special_venues = ['연회장', '라그로타']
        venue_data = df[df['store'].isin(special_venues)]
        
        print(f"      - 특수 업장 데이터: {len(venue_data)}행")
        
        if len(venue_data) == 0:
            print("      ❌ 특수 업장 데이터 없음")
            return pd.DataFrame(), np.array([]).reshape(0, 7), []
        
        valid_sequences = 0
        
        for venue in special_venues:
            venue_subset = venue_data[venue_data['store'] == venue]
            if len(venue_subset) == 0:
                continue
                
            for menu in venue_subset['menu'].unique():
                menu_data = venue_subset[venue_subset['menu'] == menu].sort_values('date').reset_index(drop=True)
                
                # 모드별 조건 분기
                if mode == 'predict':
                    if len(menu_data) < seq_length:  # 예측 모드: 28일만 필요
                        continue
                else:  # train 모드
                    if len(menu_data) < seq_length + 7:  # 훈련 모드: 35일 필요
                        continue
                
                if mode == 'predict':
                    # 예측 모드: 마지막 28일로 1개 피처만 생성
                    try:
                        seq_data = menu_data.tail(seq_length)
                        
                        # 미래 7일 정보 생성
                        last_date = seq_data['date'].iloc[-1]
                        future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=7)
                        
                        # 가상의 target_data 생성
                        target_data = pd.DataFrame({
                            'date': future_dates,
                            'month': future_dates.month,
                            'day_of_week': future_dates.dayofweek,
                            'is_weekend': future_dates.dayofweek.isin([5, 6])
                        })
                        
                        # 이벤트 기반 피처 생성
                        features = self._create_event_based_features(seq_data, target_data, venue, menu)
                        features_list.append(features)
                        venue_info_list.append({'venue': venue, 'menu': menu})
                        valid_sequences += 1
                        
                    except Exception as e:
                        continue
                        
                else:  # train 모드 - 기존 로직
                    for i in range(len(menu_data) - seq_length - 7 + 1):
                        try:
                            seq_data = menu_data.iloc[i:i+seq_length]
                            target_data = menu_data.iloc[i+seq_length:i+seq_length+7]
                            
                            # 이벤트 기반 피처 생성
                            features = self._create_event_based_features(seq_data, target_data, venue, menu)
                            features_list.append(features)
                            targets_list.append(target_data['sales'].values)
                            venue_info_list.append({'venue': venue, 'menu': menu})
                            valid_sequences += 1
                            
                        except Exception as e:
                            continue
        
        print(f"      ✅ {valid_sequences}개 시퀀스 생성")
        
        if len(features_list) == 0:
            return pd.DataFrame(), np.array([]).reshape(0, 7), []
        
        X = pd.DataFrame(features_list).fillna(0.0)
        y = np.array(targets_list) if targets_list else np.empty((0, 7))
        return X, y, venue_info_list
    
    def _create_event_based_features(self, seq_data, target_data, venue, menu):
        """이벤트 기반 특수 업장 피처"""
        sales = seq_data['sales'].values
        
        features = {
            # 극한 희소성 매출 패턴
            'total_sales': float(np.sum(sales)),
            'active_days': int(np.sum(sales > 0)),
            'active_ratio': float(np.mean(sales > 0)),
            'dormant_days': int(np.sum(sales == 0)),
            'dormant_ratio': float(np.mean(sales == 0)),
            
            # 이벤트 감지 피처
            'max_single_day': float(np.max(sales)),
            'sales_concentration': float(np.max(sales) / (np.sum(sales) + 1e-8)),
            'event_days_count': int(np.sum(sales > self.venue_config[venue]['avg_sales'])),
            'above_average_ratio': float(np.mean(sales > self.venue_config[venue]['avg_sales'])),
            
            # 이벤트 군집 분석
            'sales_bursts': int(self._detect_sales_bursts(sales, venue)),
            'burst_intensity': float(self._calculate_burst_intensity(sales)),
            'inter_event_gaps': float(self._calculate_inter_event_gaps(sales)),
            
            # 예약/이벤트 패턴 (추정)
            'weekend_event_probability': float(self._weekend_event_probability(seq_data)),
            'seasonal_event_likelihood': float(self._seasonal_event_likelihood(target_data['month'].iloc[0])),
            
            # 업장별 특수 특성
            'venue_type': venue,
            'is_banquet_hall': 1 if venue == '연회장' else 0,
            'is_specialty_restaurant': 1 if venue == '라그로타' else 0,
            
            # 메뉴 이벤트성 분석
            'is_conference_room': 1 if any(x in str(menu).lower() for x in ['conference', 'convention', 'hall']) else 0,
            'is_food_service': 1 if any(x in str(menu).lower() for x in ['밥', '찌개', '구이', '볶음']) else 0,
            'is_beverage_service': 1 if any(x in str(menu).lower() for x in ['beer', 'coffee']) else 0,
            'is_premium_menu': 1 if any(x in str(menu).lower() for x in ['갈비', 'aus', '와인']) else 0,
            
            # 시간 정보
            'target_month': int(target_data['month'].iloc[0]),
            'target_start_dow': int(target_data['day_of_week'].iloc[0]),
            'target_weekends': int(target_data['is_weekend'].sum()),
            'is_peak_month': 1 if target_data['month'].iloc[0] in [12, 1, 7, 8] else 0,
            
            # 최근 패턴 분석
            'recent_activity_level': float(np.mean(sales[-7:])),
            'momentum_score': float(self._calculate_momentum(sales)),
            'reactivation_probability': float(self._calculate_reactivation_prob(sales)),
            
            # 극값 기반 피처
            'percentile_95': float(np.percentile(sales, 95)),
            'outlier_count': int(np.sum(sales > np.percentile(sales, 95))),
        }
        
        # NaN 처리
        for key, value in features.items():
            if key not in ['venue_type'] and (pd.isna(value) or np.isinf(value)):
                features[key] = 0.0
        
        features['menu'] = str(menu)
        return features
    
    def _detect_sales_bursts(self, sales, venue):
        """매출 급증 구간 감지"""
        threshold = self.venue_config[venue]['avg_sales'] * 3
        return np.sum(sales > threshold)
    
    def _calculate_burst_intensity(self, sales):
        """급증 강도 계산"""
        if np.sum(sales) == 0:
            return 0
        return np.max(sales) / (np.mean(sales[sales > 0]) + 1e-8)
    
    def _calculate_inter_event_gaps(self, sales):
        """이벤트 간 평균 간격"""
        event_indices = np.where(sales > 0)[0]
        if len(event_indices) < 2:
            return 28
        gaps = np.diff(event_indices)
        return np.mean(gaps) if len(gaps) > 0 else 28
    
    def _weekend_event_probability(self, seq_data):
        """주말 이벤트 확률"""
        weekend_data = seq_data[seq_data['is_weekend']]
        if len(weekend_data) == 0:
            return 0
        return np.mean(weekend_data['sales'] > 0)
    
    def _seasonal_event_likelihood(self, month):
        """계절별 이벤트 가능성"""
        event_scores = {
            1: 1.8, 2: 0.8, 3: 1.2, 4: 1.1, 5: 1.3, 6: 1.0,
            7: 1.6, 8: 1.5, 9: 1.0, 10: 1.1, 11: 1.2, 12: 2.0
        }
        return event_scores.get(month, 1.0)
    
    def _calculate_momentum(self, sales):
        """최근 모멘텀 계산"""
        if len(sales) < 14:
            return 0
        recent = np.mean(sales[-7:])
        previous = np.mean(sales[-14:-7])
        return recent - previous
    
    def _calculate_reactivation_prob(self, sales):
        """재활성화 확률"""
        if len(sales) == 0:
            return 0
        recent_zeros = np.sum(sales[-7:] == 0)
        if recent_zeros == 7:
            return 0.3
        else:
            return 0.7
    
    def fit(self, X, y, venue_info):
        """✨ 누락된 fit 메서드 구현 ✨"""
        print(f"   🎭 특수 업장 모델 학습:")
        print(f"      - 피처 수: {len(X)} x {len(X.columns) if len(X) > 0 else 0}")
        print(f"      - 타겟 수: {len(y)} x {y.shape[1] if len(y) > 0 else 0}")
        
        if len(X) == 0 or len(y) == 0:
            print("      ❌ 학습 데이터 없음")
            self._is_fitted = False
            return
        
        try:
            # 피처 전처리
            X_processed = self._preprocess_special_features(X)
            
            # 이벤트 여부 분류 학습
            has_events = (y > 0).any(axis=1)
            print(f"      - 이벤트 있는 비율: {has_events.mean():.1%}")
            
            if has_events.sum() > 3:
                # 극도의 불균형 대응
                pos_weight = (len(has_events) - has_events.sum()) / max(has_events.sum(), 1)
                self.event_classifier.set_params(scale_pos_weight=min(pos_weight, 10))  # 최대 10으로 제한
                self.event_classifier.fit(X_processed, has_events)
                print("      ✅ Event classifier 학습 완료")
            else:
                print("      ⚠️ 이벤트 샘플 부족 - Event classifier 스킵")
            
            # 이벤트가 있는 경우만으로 회귀 학습
            event_mask = has_events
            if event_mask.sum() > 3:
                # 7일 중 최대값을 타겟으로 사용 (이벤트는 보통 하루에 집중)
                max_targets = y[event_mask].max(axis=1)
                
                self.conditional_regressor.fit(X_processed[event_mask], max_targets)
                print("      ✅ Conditional regressor 학습 완료")
                
                # 업장별 개별 모델 (데이터가 충분한 경우)
                for venue in ['연회장', '라그로타']:
                    venue_mask = X['venue_type'] == venue
                    combined_mask = event_mask & venue_mask
                    
                    if combined_mask.sum() > 5:
                        venue_model = LGBMRegressor(
                            n_estimators=50,
                            max_depth=3,
                            learning_rate=0.2,
                            min_child_samples=2,
                            random_state=42,
                            verbosity=-1
                        )
                        venue_targets = y[combined_mask].max(axis=1)
                        venue_model.fit(X_processed[combined_mask], venue_targets)
                        self.venue_specific_models[venue] = venue_model
                        print(f"      ✅ {venue} 전용 모델 학습 완료")
            
            self._is_fitted = True
            print("      ✅ 전체 학습 완료")
            
        except Exception as e:
            print(f"      ❌ 학습 오류: {e}")
            self._is_fitted = False
    
    def _preprocess_special_features(self, X):
        """특수 업장 피처 전처리"""
        X_processed = X.copy()
        
        # 업장 인코딩
        venue_map = {'연회장': 1, '라그로타': 0}
        if 'venue_type' in X_processed.columns:
            X_processed['venue_encoded'] = X_processed['venue_type'].map(venue_map).fillna(0)
        
        # 메뉴 해시 인코딩
        if 'menu' in X_processed.columns:
            X_processed['menu_hash'] = X_processed['menu'].apply(lambda x: hash(str(x)) % 50)
            X_processed = X_processed.drop('menu', axis=1)
        
        # venue_type 제거 (인코딩 완료)
        if 'venue_type' in X_processed.columns:
            X_processed = X_processed.drop('venue_type', axis=1)
        
        # 숫자형만 선택
        numeric_features = X_processed.select_dtypes(include=[np.number]).columns
        X_numeric = X_processed[numeric_features].fillna(0)
        X_numeric = X_numeric.replace([np.inf, -np.inf], 0)
        
        return X_numeric
    
    def predict(self, X):
        """✨ 누락된 predict 메서드 구현 ✨"""
        if not getattr(self, '_is_fitted', False):
            print("      ⚠️ 모델이 학습되지 않음 - 기본값 반환")
            return np.full((len(X), 7), 1.0)
        
        try:
            X_processed = self._preprocess_special_features(X)
            predictions = np.zeros((len(X), 7))
            
            # 이벤트 확률 예측
            try:
                event_prob = self.event_classifier.predict_proba(X_processed)[:, 1]
            except:
                event_prob = np.full(len(X), 0.1)  # 낮은 기본값
            
            # 조건부 매출량 예측
            try:
                conditional_sales = self.conditional_regressor.predict(X_processed)
            except:
                conditional_sales = np.array([self.venue_config.get(venue, {}).get('avg_sales', 2.0) 
                                           for venue in X.get('venue_type', ['연회장'] * len(X))])
            
            # 업장별 예측 조정
            for i in range(len(X)):
                venue = X.iloc[i].get('venue_type', '연회장')
                base_pred = conditional_sales[i] * event_prob[i]
                
                # 업장별 특화 모델 사용
                if venue in self.venue_specific_models:
                    try:
                        venue_pred = self.venue_specific_models[venue].predict(X_processed[i:i+1])[0]
                        base_pred = 0.6 * base_pred + 0.4 * venue_pred * event_prob[i]
                    except:
                        pass
                
                # 이벤트는 보통 1-2일에 집중
                if event_prob[i] > 0.3:  # 이벤트 가능성이 높은 경우
                    # 랜덤하게 1-2일에 집중
                    event_days = np.random.choice(7, size=np.random.randint(1, 3), replace=False)
                    for day in event_days:
                        predictions[i, day] = base_pred
                else:
                    # 낮은 확률로 분산
                    predictions[i] = base_pred * 0.3
            
            # 음수 방지
            predictions = np.clip(predictions, 0, None)
            
            return predictions
            
        except Exception as e:
            print(f"      ❌ 예측 오류: {e}")
            # 업장별 기본값 반환
            default_values = []
            for i in range(len(X)):
                venue = X.iloc[i].get('venue_type', '연회장')
                default_val = self.venue_config.get(venue, {}).get('avg_sales', 2.0)
                default_values.append([default_val * 0.5] * 7)  # 보수적 예측
            
            return np.array(default_values)
# 모델 사용 예시
def train_outlet_and_venue_models(train_df):
    # 일반 업장 모델
    regular_model = RegularOutletModel()
    X_regular, y_regular = regular_model.create_regular_outlet_features(train_df)
    regular_model.fit(X_regular, y_regular)
    
    # 특수 업장 모델
    special_model = SpecialVenueModel()
    X_special, y_special, venue_info = special_model.create_special_venue_features(train_df)
    special_model.fit(X_special, y_special, venue_info)
    
    return regular_model, special_model

# 예측
def predict_outlets_and_venues(regular_model, special_model, test_data):
    # 일반 업장 예측
    X_regular, _ = regular_model.create_regular_outlet_features(test_data)
    regular_pred = regular_model.predict(X_regular)
    
    # 특수 업장 예측
    X_special, _, _ = special_model.create_special_venue_features(test_data)
    special_pred = special_model.predict(X_special)
    
    return regular_pred, special_pred

# ----------------------------
# Ensemble System
# ----------------------------
import re  # 정규표현식을 위해 추가
class IntegratedForecastingSystem:
    def __init__(self):
        # 기본 설정은 동일
        self.models = {
            'damha_individual': DamhaMLModel(),
            'mirasia_individual': MirasiaMLModel(), 
            'main_restaurants': MainRestaurantModel(),
            'regular_outlets': RegularOutletModel(),
            'special_venues': SpecialVenueModel()
        }
        
        self.store_category_mapping = {
            '담하': 'damha_individual',
            '미라시아': 'mirasia_individual',
            '포레스트릿': 'main_restaurants',
            '카페테리아': 'main_restaurants', 
            '화담숲주막': 'main_restaurants',
            '화담숲카페': 'main_restaurants',
            '느티나무 셀프BBQ': 'regular_outlets',
            '연회장': 'special_venues',
            '라그로타': 'special_venues'
        }
        
        # 최소 데이터 요구사항
        self.min_data_requirements = {
            'damha_individual': 50,
            'mirasia_individual': 50,
            'main_restaurants': 100,
            'regular_outlets': 30,
            'special_venues': 20
        }
        
    def fit(self, train_df: pd.DataFrame, validation_df: pd.DataFrame = None):
        """수정된 전체 시스템 학습"""
        print("🚀 통합 예측 시스템 학습 시작...")
        
        category_performance = {}
        
        for category, model in self.models.items():
            print(f"\n📊 {category} 모델 학습 중...")
            
            try:
                # 카테고리별 데이터 필터링
                category_stores = [store for store, cat in self.store_category_mapping.items() if cat == category]
                category_data = train_df[train_df['store'].isin(category_stores)].copy()
                if len(category_data) == 0:
                    print(f"⚠️ {category}: 데이터 없음")
                    continue
                
                # 최소 데이터 요구사항 확인
                min_required = self.min_data_requirements.get(category, 20)
                if len(category_data) < min_required:
                    print(f"⚠️ {category}: 데이터 부족 ({len(category_data)} < {min_required})")
                    continue
                
                # 모델별 피처 생성 및 학습 (수정된 버전)
                success = self._train_category_model_safe(model, category, category_data)
                
                if success:
                    print(f"✅ {category} 학습 완료")
                    
                    # 검증 데이터로 성능 평가 (안전한 버전)
                    if validation_df is not None:
                        val_performance = self._validate_category_model_safe(
                            model, category, validation_df
                        )
                        category_performance[category] = val_performance
                else:
                    print(f"❌ {category} 학습 실패")
                    
            except Exception as e:
                print(f"❌ {category} 학습 중 오류: {str(e)}")
                continue
        
        # 성능 요약 출력
        if category_performance:
            self._print_performance_summary(category_performance)
        
        print("\n🎉 전체 시스템 학습 완료!")
    
    def _train_category_model_safe(self, model, category: str, category_data: pd.DataFrame) -> bool:
        """안전한 카테고리 모델 학습"""
        try:
            if category == 'damha_individual':
                X, y = model.create_damha_features(category_data)
        
                if len(X) > 10 and len(y) > 10:  # 최소 데이터 확인
                    model.fit(X, y)
                    return True
                    
            elif category == 'mirasia_individual':
                X, y = model.create_mirasia_features(category_data)
                
                if len(X) > 10 and len(y) > 10:
                    model.fit(X, y)
                    return True
                    
            elif category == 'main_restaurants':
                X, y, store_info = model.create_main_restaurant_features(category_data)
                
                if len(X) > 20 and len(y) > 20:
                    model.fit(X, y, store_info)
                    return True
                    
            elif category == 'regular_outlets':
                X, y = model.create_regular_outlet_features(category_data)
                
                if len(X) > 10 and len(y) > 10:
                    model.fit(X, y)
                    return True
                    
            elif category == 'special_venues':
                X, y, venue_info = model.create_special_venue_features(category_data)
                
                if len(X) > 5 and len(y) > 5:
                    model.fit(X, y, venue_info)
                    return True
            
            return False
            
        except Exception as e:
            print(f"   오류: {str(e)}")
            return False
        
    def _predict_category_complete(self, model, category: str, category_data: pd.DataFrame):
        """수정된 카테고리별 예측"""
        try:
            print(f"   {category} 예측 시도: {len(category_data)}행")
            
            if category == 'damha_individual':
                X_test, _ = model.create_damha_features(category_data, mode='predict')
                
            elif category == 'mirasia_individual':
                X_test, _ = model.create_mirasia_features(category_data, mode='predict')  # 올바른 메서드명
                
            elif category == 'main_restaurants':
                X_test, _, _ = model.create_main_restaurant_features(category_data, mode='predict')
                
            elif category == 'regular_outlets':
                X_test, _ = model.create_regular_outlet_features(category_data, mode='predict')
                
            elif category == 'special_venues':
                X_test, _, _ = model.create_special_venue_features(category_data, mode='predict')
                
            else:
                print(f"   알 수 없는 카테고리: {category}")
                return None
                
            print(f"   피처 생성: {X_test.shape}")
            
            if len(X_test) > 0:
                predictions = model.predict(X_test)  # 모든 모델이 predict 메서드 사용
                print(f"   예측 완료: {predictions.shape}")
                return predictions
            else:
                print(f"   피처 없음")
                return None
                
        except Exception as e:
            print(f"   예측 오류: {str(e)}")
            import traceback
            traceback.print_exc()  # 상세한 에러 정보 출력
            return None

    def predict(self, test_df: pd.DataFrame) -> Dict[str, np.ndarray]:
        """수정된 예측 메서드"""
        print("통합 예측 시스템 예측 시작...")
        
        all_predictions = {}
        
        for category, model in self.models.items():
            print(f"{category} 예측 중...")
            
            try:
                # 카테고리별 데이터 추출
                category_stores = [store for store, cat in self.store_category_mapping.items() 
                                if cat == category]
                category_data = test_df[test_df['store'].isin(category_stores)].copy()
                
                print(f"   업장: {category_stores}")
                print(f"   데이터: {len(category_data)}행")
                
                if len(category_data) == 0:
                    print(f"   테스트 데이터 없음")
                    continue
                
                # 예측 수행
                predictions = self._predict_category_complete(model, category, category_data)
                
                if predictions is not None and len(predictions) > 0:
                    all_predictions[category] = predictions
                    print(f"   성공: {predictions.shape}")
                else:
                    print(f"   실패")
                    
            except Exception as e:
                print(f"   {category} 예측 오류: {str(e)}")
                import traceback
                traceback.print_exc()
        
        print(f"예측 완료! 성공한 카테고리: {len(all_predictions)}개")
        
        if not all_predictions:
            print("모든 예측 실패 - 기본값 반환")
            all_predictions = self._create_default_predictions(test_df)
            
        all_predictions = clip_negative_predictions(all_predictions)
        return all_predictions
        
    def _create_default_predictions(self, test_df: pd.DataFrame) -> Dict[str, np.ndarray]:
        """기본 예측값 생성 (모든 모델 실패시)"""
        print("🛡️ 기본 예측값 생성 중...")
        
        default_predictions = {}
        
        # 업장별 기본 예측값 (과거 분석 기반)
        store_defaults = {
            '담하': 6.0,
            '미라시아': 6.0, 
            '포레스트릿': 48.0,
            '카페테리아': 19.0,
            '화담숲주막': 34.0,
            '화담숲카페': 24.0,
            '느티나무 셀프BBQ': 6.0,
            '연회장': 2.0,
            '라그로타': 1.0
        }
        
        for category, stores in [
            ('damha_individual', ['담하']),
            ('mirasia_individual', ['미라시아']),
            ('main_restaurants', ['포레스트릿', '카페테리아', '화담숲주막', '화담숲카페']),
            ('regular_outlets', ['느티나무 셀프BBQ']),
            ('special_venues', ['연회장', '라그로타'])
        ]:
            category_data = test_df[test_df['store'].isin(stores)]
            
            if len(category_data) > 0:
                # 업장별 기본값으로 예측 생성
                predictions = []
                for _, row in category_data.iterrows():
                    store = row['store']
                    default_value = store_defaults.get(store, 5.0)
                    # 7일 동일값
                    predictions.append([default_value] * 7)
                
                default_predictions[category] = np.array(predictions)
                print(f"   {category}: {len(predictions)}개 기본 예측 생성")
        
        return default_predictions

    
    def _validate_category_model_safe(self, model, category: str, validation_df: pd.DataFrame) -> Dict:
        """안전한 카테고리별 모델 검증"""
        try:
            category_stores = [store for store, cat in self.store_category_mapping.items() if cat == category]
            val_data = validation_df[validation_df['store'].isin(category_stores)]
            
            if len(val_data) == 0:
                return {'smape': float('inf'), 'mae': float('inf'), 'status': 'no_data'}
            
            # 검증 데이터로 피처 생성 (모델별로)
            if category == 'damha_individual':
                X_val, y_val = model.create_damha_features(val_data)
            elif category == 'mirasia_individual':
                X_val, y_val = model.create_mirasia_features(val_data)
            elif category == 'main_restaurants':
                X_val, y_val, _ = model.create_main_restaurant_features(val_data)
            elif category == 'regular_outlets':
                X_val, y_val = model.create_regular_outlet_features(val_data)
            elif category == 'special_venues':
                X_val, y_val, _ = model.create_special_venue_features(val_data)
            else:
                return {'smape': float('inf'), 'mae': float('inf'), 'status': 'unknown_category'}
            
            if len(X_val) == 0 or len(y_val) == 0:
                return {'smape': float('inf'), 'mae': float('inf'), 'status': 'insufficient_validation_data'}
            
            # 예측 수행
            predictions = model.predict(X_val)
            
            # 성능 계산
            smape = self._calculate_smape_safe(y_val, predictions)
            mae = np.mean(np.abs(y_val - predictions)) if y_val.size > 0 and predictions.size > 0 else float('inf')
            
            return {
                'smape': smape, 
                'mae': mae, 
                'status': 'success',
                'validation_samples': len(X_val)
            }
            
        except Exception as e:
            print(f"   검증 오류: {str(e)}")
            return {'smape': float('inf'), 'mae': float('inf'), 'status': f'error_{str(e)[:50]}'}
    
    def _calculate_smape_safe(self, actual: np.ndarray, predicted: np.ndarray) -> float:
        """안전한 SMAPE 계산"""
        try:
            # 배열 형태 통일
            if actual.ndim > 1:
                actual = actual.flatten()
            if predicted.ndim > 1:
                predicted = predicted.flatten()
            
            # 크기 맞춤
            min_size = min(len(actual), len(predicted))
            actual = actual[:min_size]
            predicted = predicted[:min_size]
            
            if len(actual) == 0:
                return float('inf')
            
            # 실제 매출이 0이 아닌 경우만 계산
            mask = (actual > 0) | (predicted > 0)  # 둘 중 하나라도 0이 아니면 포함
            
            actual_filtered = actual[mask]
            predicted_filtered = predicted[mask]
            
            denominator = (np.abs(actual_filtered) + np.abs(predicted_filtered)) / 2
            denominator = np.maximum(denominator, 1e-8)  # 0으로 나누기 방지
            
            smape = np.mean(np.abs(actual_filtered - predicted_filtered) / denominator) * 100
            
            return float(smape) if not np.isnan(smape) else float('inf')
            
        except Exception as e:
            print(f"   SMAPE 계산 오류: {str(e)}")
            return float('inf')
    
    def _print_performance_summary(self, category_performance: Dict):
        """성능 요약 출력"""
        print("\n📊 모델 성능 요약:")
        print("-" * 50)
        
        for category, metrics in category_performance.items():
            status = metrics.get('status', 'unknown')
            smape = metrics.get('smape', float('inf'))
            mae = metrics.get('mae', float('inf'))
            samples = metrics.get('validation_samples', 0)
            
            smape_str = f"{smape:.2f}%" if smape != float('inf') else "∞"
            mae_str = f"{mae:.2f}" if mae != float('inf') else "∞"
            
            print(f"{category:20} | SMAPE: {smape_str:8} | MAE: {mae_str:8} | 샘플: {samples:4} | {status}")
    def _check_data_quality(self, X: pd.DataFrame, y: np.ndarray, category: str):
        """데이터 품질 체크"""
        print(f"   📊 {category} 데이터 품질 체크:")
        print(f"      - 샘플 수: {len(X)}")
        print(f"      - 피처 수: {len(X.columns)}")
        print(f"      - 0매출 비율: {(y == 0).mean():.1%}")
        print(f"      - 평균 매출: {y.mean():.2f}")
        
        # 피처별 분산 체크
        low_variance_features = (X.var() < 1e-6).sum()
        if low_variance_features > 0:
            print(f"      - 저분산 피처: {low_variance_features}개 (제거됨)")
        
        # 결측값 체크
        missing_ratio = X.isnull().mean().mean()
        if missing_ratio > 0:
            print(f"      - 결측값 비율: {missing_ratio:.1%}")
        
        return True

class MetaEnsembleModel:
    def __init__(self):
        from sklearn.ensemble import RandomForestRegressor
        from xgboost import XGBRegressor
        
        # 메타 모델들
        self.meta_models = {
            'rf': RandomForestRegressor(n_estimators=100, random_state=42),
            'xgb': XGBRegressor(n_estimators=100, random_state=42),
            'weighted_avg': None  # 가중 평균
        }
        
        # 카테고리별 가중치 (성능 기반으로 동적 조정)
        self.category_weights = {
            'damha_individual': 1.0,
            'mirasia_individual': 1.0,
            'main_restaurants': 1.0,
            'regular_outlets': 1.0,
            'special_venues': 1.0
        }
        
    def fit(self, base_predictions: Dict, actual_values: np.ndarray, store_info: List):
        """메타 모델 학습"""
        # 기본 모델들의 예측을 피처로 사용
        meta_features = self._create_meta_features(base_predictions, store_info)
        
        # 메타 모델 학습
        for name, model in self.meta_models.items():
            if model is not None:
                model.fit(meta_features, actual_values.flatten())
    
    def predict(self, base_predictions: Dict, store_info: List) -> np.ndarray:
        """메타 모델 예측"""
        meta_features = self._create_meta_features(base_predictions, store_info)
        
        # 각 메타 모델의 예측을 앙상블
        meta_predictions = []
        for name, model in self.meta_models.items():
            if model is not None:
                pred = model.predict(meta_features)
                meta_predictions.append(pred)
        
        # 메타 모델들의 평균
        return np.mean(meta_predictions, axis=0)
    
    def _create_meta_features(self, base_predictions: Dict, store_info: List) -> np.ndarray:
        """메타 피처 생성"""
        features = []
        
        # 기본 모델 예측값들을 피처로 사용
        for category, predictions in base_predictions.items():
            features.append(predictions.flatten())
        
        # 업장 정보를 추가 피처로 사용
        # store_weights, category 등
        
        return np.column_stack(features)

class DynamicWeightingSystem:
    def __init__(self):
        # 성능 기반 동적 가중치 시스템
        self.performance_history = {}
        self.weight_decay = 0.9  # 과거 성능의 감쇠율
        self.min_weight = 0.1    # 최소 가중치
        
    def update_weights(self, category_performance: Dict) -> Dict:
        """성능 기반 가중치 업데이트"""
        updated_weights = {}
        
        for category, performance in category_performance.items():
            current_smape = performance.get('smape', float('inf'))
            
            # 성능이 좋을수록 높은 가중치
            if current_smape == float('inf'):
                weight = self.min_weight
            else:
                # SMAPE가 낮을수록 높은 가중치 (역수 관계)
                weight = 1.0 / (1.0 + current_smape / 100)
                weight = max(weight, self.min_weight)
            
            # 과거 성능과 가중 평균
            if category in self.performance_history:
                previous_weight = self.performance_history[category]
                weight = self.weight_decay * previous_weight + (1 - self.weight_decay) * weight
            
            updated_weights[category] = weight
            self.performance_history[category] = weight
        
        # 정규화
        total_weight = sum(updated_weights.values())
        if total_weight > 0:
            updated_weights = {k: v/total_weight for k, v in updated_weights.items()}
        
        return updated_weights

    def _train_meta_ensemble(self, validation_df: pd.DataFrame):
        """메타 앙상블 모델 학습"""
        try:
            # 검증 데이터로 기본 모델들의 예측 수집
            base_predictions = {}
            actual_values = []
            store_info = []
            
            for category, model in self.models.items():
                category_stores = [store for store, cat in self.store_category_mapping.items() if cat == category]
                val_data = validation_df[validation_df['store'].isin(category_stores)]
                
                if len(val_data) == 0:
                    continue
                
                # 카테고리별 예측 및 실제값 수집
                # ... (각 모델별 예측 코드)
            
            # 메타 모델 초기화 및 학습
            self.meta_model = MetaEnsembleModel()
            if base_predictions and len(actual_values) > 0:
                self.meta_model.fit(base_predictions, np.array(actual_values), store_info)
                self.use_meta_ensemble = True
                print("✅ 메타 앙상블 모델 학습 완료")
            
        except Exception as e:
            print(f"❌ 메타 앙상블 학습 실패: {str(e)}")
            self.use_meta_ensemble = False
    
    def _postprocess_predictions(self, predictions: Dict, test_df: pd.DataFrame) -> Dict:
        """예측 후처리 및 보정"""
        final_predictions = {}
        
        for category, preds in predictions.items():
            # 1. 음수 제거
            preds = np.maximum(preds, 0)
            
            # 2. 업장별 스케일 보정
            category_stores = [store for store, cat in self.store_category_mapping.items() if cat == category]
            
            for store in category_stores:
                store_mask = test_df['store'] == store
                if store_mask.any():
                    # 업장별 히스토리컬 평균으로 보정
                    store_correction = self._get_store_correction_factor(store)
                    store_indices = np.where(store_mask)[0]
                    preds[store_indices] *= store_correction
            
            # 3. 시계열 일관성 체크
            preds = self._ensure_temporal_consistency(preds)
            
            final_predictions[category] = preds
        
        return final_predictions
    
    def _get_store_correction_factor(self, store: str) -> float:
        """업장별 보정 팩터"""
        base_factors = {
            '담하': 1.0,
            '미라시아': 1.0, 
            '포레스트릿': 1.2,  # 안정적 고매출
            '카페테리아': 1.1,
            '화담숲주막': 1.15,
            '화담숲카페': 1.05,
            '느티나무 셀프BBQ': 0.95,  # 변동성 고려
            '연회장': 0.8,     # 불확실성 높음
            '라그로타': 0.8
        }
        return base_factors.get(store, 1.0)
    
    def _ensure_temporal_consistency(self, predictions: np.ndarray) -> np.ndarray:
        """시계열 일관성 보장"""
        # 급격한 변화 완화
        if predictions.shape[1] == 7:  # 7일 예측
            for i in range(len(predictions)):
                for day in range(1, 7):
                    # 전날 대비 급격한 변화 제한 (최대 300% 변화)
                    max_change_ratio = 3.0
                    prev_day = predictions[i, day-1]
                    current_day = predictions[i, day]
                    
                    if prev_day > 0:
                        change_ratio = current_day / prev_day
                        if change_ratio > max_change_ratio:
                            predictions[i, day] = prev_day * max_change_ratio
                        elif change_ratio < 1/max_change_ratio:
                            predictions[i, day] = prev_day / max_change_ratio
        
        return predictions

class ModelPerformanceTracker:
    def __init__(self):
        self.performance_history = {}
        self.current_metrics = {}
        
    def log_performance(self, category: str, metrics: Dict):
        """성능 로깅"""
        if category not in self.performance_history:
            self.performance_history[category] = []
        
        self.performance_history[category].append(metrics)
        self.current_metrics[category] = metrics
    
    def get_performance_summary(self) -> Dict:
        """성능 요약 반환"""
        summary = {}
        for category, metrics in self.current_metrics.items():
            summary[category] = {
                'current_smape': metrics.get('smape', 'N/A'),
                'current_mae': metrics.get('mae', 'N/A'),
                'trend': self._calculate_trend(category)
            }
        return summary
    
    def _calculate_trend(self, category: str) -> str:
        """성능 트렌드 계산"""
        if category not in self.performance_history or len(self.performance_history[category]) < 2:
            return "insufficient_data"
        
        recent_smape = self.performance_history[category][-1].get('smape', float('inf'))
        previous_smape = self.performance_history[category][-2].get('smape', float('inf'))
        
        if recent_smape < previous_smape:
            return "improving"
        elif recent_smape > previous_smape:
            return "degrading"
        else:
            return "stable"

# ----------------------------
# submission format
# ----------------------------

# ----------------------------
# Main pipeline
# ----------------------------
def main_prediction_pipeline():
    """전체 예측 파이프라인 실행"""
    
    logging.debug("\n🚀 리조트 식음업장 수요 예측 시스템 시작")
    logging.debug("=" * 50)
    
    # 1. 데이터 로드 및 전처리
    logging.debug("\n📊 1단계: 데이터 로드")
    train_df = load_and_preprocess_data('./train/train.csv')
    

    # 검증 데이터 분리 (최근 1개월)
    latest_date = train_df['date'].max()
    validation_cutoff = latest_date - pd.Timedelta(days=30)
    
    train_data = train_df[train_df['date'] <= validation_cutoff]
    validation_data = train_df[train_df['date'] > validation_cutoff]
    
    logging.debug(f"   - 훈련 데이터: {len(train_data):,}행")
    logging.debug(f"   - 검증 데이터: {len(validation_data):,}행")
    
    # 2. 통합 예측 시스템 초기화 및 학습
    logging.debug("\n🤖 2단계: 모델 학습")
    forecasting_system = IntegratedForecastingSystem()
    forecasting_system.fit(train_data, validation_data)
    forecasting_system.use_meta_ensemble = True  # 메타 앙상블 활성화
    
    # 3. 제출 파일 생성
    logging.debug("\n📋 3단계: 제출 파일 생성")
    # 실행
    logging.debug("메뉴별 개별 예측값으로 제출 파일 생성 중...")
    submission_df = create_menu_specific_submission(forecasting_system)

    # 검증
    logging.debug(f"\n제출 파일 검증:")
    logging.debug(f"형태: {submission_df.shape}")
    non_zero_count = (submission_df.select_dtypes(include=[np.number]) != 0).sum().sum()
    logging.debug(f"0이 아닌 값: {non_zero_count}개")

    # 담하 컬럼에서 값의 다양성 확인
    damha_cols = [col for col in submission_df.columns if '담하_' in col][:10]
    logging.debug(f"\n담하 첫 10개 컬럼 (첫 7행):")
    damha_sample = submission_df[damha_cols].head(7)
    logging.debug(damha_sample)

    # 같은 행에서 값이 다른지 확인
    logging.debug(f"\n첫 번째 행의 고유값 개수: {damha_sample.iloc[0].nunique()}")
    logging.debug(f"두 번째 행의 고유값 개수: {damha_sample.iloc[1].nunique()}")

    # 저장
    submission_df.to_csv('menu_specific_submission.csv', index=False)
    logging.debug("파일 저장: menu_specific_submission.csv")
    
    return submission_df

def create_menu_specific_submission(forecasting_system):
    """메뉴별 개별 예측값을 사용한 제출 파일 생성"""
    
    submission = pd.read_csv('sample_submission.csv')
    
    # 각 테스트 파일별로 처리
    test_files = sorted(glob.glob('TEST_*.csv'))
    
    for test_idx, test_file in enumerate(test_files):
        print(f"처리 중: {test_file}")
        
        # 테스트 데이터 로드
        test_df = load_and_preprocess_data(test_file)
        
        # 예측 수행
        predictions = forecasting_system.predict(test_df)
        
        # 메뉴 정보도 함께 추출
        menu_info = extract_menu_info_from_predictions(test_df, predictions)
        
        # 제출 파일 매핑
        test_case = f"TEST_{test_idx:02d}"
        test_rows = submission[submission['영업일자'].str.contains(test_case, na=False)].index.tolist()
        
        if len(test_rows) != 7:
            continue
            
        # 카테고리별로 메뉴와 예측값 매핑
        map_predictions_to_submission(submission, test_rows, predictions, menu_info)
    
    return submission

def extract_menu_info_from_predictions(test_df, predictions):
    """예측 결과와 메뉴 정보 매핑"""
    menu_info = {}
    
    for category, pred_array in predictions.items():
        category_stores = get_stores_by_category(category)
        category_data = test_df[test_df['store'].isin(category_stores)]
        
        # 각 카테고리별 메뉴 순서 추출
        menu_list = []
        for store in category_stores:
            store_data = category_data[category_data['store'] == store]
            for menu in store_data['menu'].unique():
                menu_data = store_data[store_data['menu'] == menu]
                if len(menu_data) >= 28:  # 충분한 데이터가 있는 메뉴만
                    menu_list.append((store, menu))
        
        menu_info[category] = menu_list[:len(pred_array)]  # 예측 배열 크기와 맞춤
    
    return menu_info

def map_predictions_to_submission(submission, test_rows, predictions, menu_info):
    """예측값을 제출 파일에 메뉴별로 매핑"""
    
    for category, pred_array in predictions.items():
        if category not in menu_info:
            continue
            
        category_menu_list = menu_info[category]
        
        for menu_idx, (store, menu) in enumerate(category_menu_list):
            if menu_idx >= len(pred_array):
                break
                
            # 제출 파일에서 해당 메뉴 컬럼 찾기
            target_column = f"{store}_{menu}"
            
            if target_column in submission.columns:
                # 7일간 개별 예측값 할당
                for day_idx, row_idx in enumerate(test_rows):
                    if day_idx < pred_array.shape[1]:
                        individual_prediction = pred_array[menu_idx, day_idx]
                        submission.loc[row_idx, target_column] = max(0, int(round(individual_prediction)))
            else:
                # 정확한 컬럼명을 찾지 못한 경우, 유사한 컬럼 찾기
                similar_columns = [col for col in submission.columns 
                                 if col.startswith(f"{store}_") and menu in col]
                
                if similar_columns:
                    for day_idx, row_idx in enumerate(test_rows):
                        if day_idx < pred_array.shape[1]:
                            individual_prediction = pred_array[menu_idx, day_idx]
                            # 유사한 컬럼들에 동일한 값 할당
                            for col in similar_columns:
                                submission.loc[row_idx, col] = max(0, int(round(individual_prediction)))

def get_stores_by_category(category):
    """카테고리별 업장 반환"""
    mapping = {
        'damha_individual': ['담하'],
        'mirasia_individual': ['미라시아'], 
        'main_restaurants': ['포레스트릿', '카페테리아', '화담숲주막', '화담숲카페'],
        'regular_outlets': ['느티나무 셀프BBQ'],
        'special_venues': ['연회장', '라그로타']
    }
    return mapping.get(category, [])
# 실행
final_submission = main_prediction_pipeline()

# 디버깅 코드
def debug_prediction_pipeline(forecasting_system):
    # 1. 모델 학습 상태 확인
    logging.debug("=== 모델 학습 상태 확인 ===")
    for category, model in forecasting_system.models.items():
        fitted = getattr(model, '_is_fitted', False)
        logging.debug(f"{category}: 학습됨={fitted}")
    
    # 2. 테스트 파일 확인
    test_files = glob.glob('TEST_*.csv')
    logging.debug(f"\n=== 테스트 파일: {len(test_files)}개 ===")
    for f in test_files[:3]:  # 처음 3개만 확인
        df = pd.read_csv(f)
        logging.debug(f"{f}: {df.shape}, 매출수량 범위: [{df['매출수량'].min()}, {df['매출수량'].max()}]")
    
    # 3. 단일 파일로 예측 테스트
    test_df = load_and_preprocess_data('TEST_00.csv')
    predictions = forecasting_system.predict(test_df)
    
    logging.debug(f"\n=== 예측 결과 ===")
    for category, pred in predictions.items():
        logging.debug(f"{category}: {pred.shape}, 범위: [{pred.min():.2f}, {pred.max():.2f}]")
    
    return predictions
# 예측 후 음수값 클리핑
def clip_negative_predictions(predictions):
    """음수 예측값 제거"""
    clipped = {}
    for category, pred_array in predictions.items():
        clipped[category] = np.clip(pred_array, 0, None)
        negative_count = (pred_array < 0).sum()
        if negative_count > 0:
            print(f"{category}: {negative_count}개 음수값 제거됨")
    return clipped

2025-08-21 15:23:15,207 DEBUG 
🚀 리조트 식음업장 수요 예측 시스템 시작
2025-08-21 15:23:15,208 DEBUG ==================================================
2025-08-21 15:23:15,208 DEBUG 
📊 1단계: 데이터 로드
2025-08-21 15:23:15,725 DEBUG    - 훈련 데이터: 88,713행
2025-08-21 15:23:15,726 DEBUG    - 검증 데이터: 5,790행
2025-08-21 15:23:15,726 DEBUG 
🤖 2단계: 모델 학습


🚀 통합 예측 시스템 학습 시작...

📊 damha_individual 모델 학습 중...
✅ damha_individual 학습 완료

📊 mirasia_individual 모델 학습 중...
✅ mirasia_individual 학습 완료

📊 main_restaurants 모델 학습 중...
✅ main_restaurants 학습 완료

📊 regular_outlets 모델 학습 중...
   📊 일반 업장 피처 생성:
      - 입력 데이터: 11546행
      - 느티나무 셀프BBQ 데이터: 11546행
      ✅ 10764개 시퀀스 생성
   📚 일반 업장 모델 학습:
      - 피처 수: 10764 x 31
      - 타겟 수: 10764 x 7
      - 매출 있는 비율: 80.9%
      ✅ Zero classifier 학습 완료
      ✅ Sales regressor 학습 완료
      ✅ 일자별 모델 7개 학습 완료
      ✅ 전체 학습 완료
✅ regular_outlets 학습 완료
   📊 일반 업장 피처 생성:
      - 입력 데이터: 690행
      - 느티나무 셀프BBQ 데이터: 690행
      ✅ 0개 시퀀스 생성

📊 special_venues 모델 학습 중...
   🎭 특수 업장 피처 생성:
      - 특수 업장 데이터: 21137행
      ✅ 19505개 시퀀스 생성
   🎭 특수 업장 모델 학습:
      - 피처 수: 19505 x 31
      - 타겟 수: 19505 x 7
      - 이벤트 있는 비율: 86.1%
      ✅ Event classifier 학습 완료


2025-08-21 15:24:49,338 DEBUG 
📋 3단계: 제출 파일 생성
2025-08-21 15:24:49,340 DEBUG 메뉴별 개별 예측값으로 제출 파일 생성 중...


      ✅ Conditional regressor 학습 완료
      ✅ 연회장 전용 모델 학습 완료
      ✅ 라그로타 전용 모델 학습 완료
      ✅ 전체 학습 완료
✅ special_venues 학습 완료
   🎭 특수 업장 피처 생성:
      - 특수 업장 데이터: 1440행
      ✅ 0개 시퀀스 생성

📊 모델 성능 요약:
--------------------------------------------------
damha_individual     | SMAPE: ∞        | MAE: ∞        | 샘플:    0 | insufficient_validation_data
mirasia_individual   | SMAPE: ∞        | MAE: ∞        | 샘플:    0 | insufficient_validation_data
main_restaurants     | SMAPE: ∞        | MAE: ∞        | 샘플:    0 | insufficient_validation_data
regular_outlets      | SMAPE: ∞        | MAE: ∞        | 샘플:    0 | insufficient_validation_data
special_venues       | SMAPE: ∞        | MAE: ∞        | 샘플:    0 | insufficient_validation_data

🎉 전체 시스템 학습 완료!
처리 중: TEST_00.csv
통합 예측 시스템 예측 시작...
damha_individual 예측 중...
   업장: ['담하']
   데이터: 1176행
   damha_individual 예측 시도: 1176행
   피처 생성: (42, 40)
   예측 완료: (42, 7)
   성공: (42, 7)
mirasia_individual 예측 중...
   업장: ['미라시아']
   데이터: 868행
   mirasia_indivi

2025-08-21 15:24:54,758 DEBUG 
제출 파일 검증:
2025-08-21 15:24:54,758 DEBUG 형태: (70, 194)
2025-08-21 15:24:54,759 DEBUG 0이 아닌 값: 9319개
2025-08-21 15:24:54,760 DEBUG 
담하 첫 10개 컬럼 (첫 7행):
2025-08-21 15:24:54,760 DEBUG    담하_(단체) 공깃밥  담하_(단체) 생목살 김치전골 2.0  담하_(단체) 은이버섯 갈비탕  담하_(단체) 한우 우거지 국밥  \
0            2                    20                17                  3   
1            2                    20                17                  2   
2            2                    20                19                  2   
3            3                    19                18                  2   
4            2                    19                20                  2   
5            3                    23                22                  3   
6            2                    27                26                  2   

   담하_(단체) 황태해장국 3/27까지  담하_(정식) 된장찌개  담하_(정식) 물냉면   담하_(정식) 비빔냉면  \
0                    30             6             3             2   
1                    38           

In [ ]:
print("🚀 리조트 식음업장 수요 예측 시스템 시작")
print("=" * 50)

# 1. 데이터 로드 및 전처리
print("\n📊 1단계: 데이터 로드")
train_df = load_and_preprocess_data('./train/train.csv')


# 검증 데이터 분리 (최근 1개월)
latest_date = train_df['date'].max()
validation_cutoff = latest_date - pd.Timedelta(days=30)

train_data = train_df[train_df['date'] <= validation_cutoff]
validation_data = train_df[train_df['date'] > validation_cutoff]

logging.debug(f"   - 훈련 데이터: {len(train_data):,}행")
logging.debug(f"   - 검증 데이터: {len(validation_data):,}행")
forecasting_system = IntegratedForecastingSystem()
forecasting_system.fit(train_data, validation_data)
# 실행
debug_results = debug_prediction_pipeline(forecasting_system)

🚀 리조트 식음업장 수요 예측 시스템 시작

📊 1단계: 데이터 로드


2025-08-21 14:50:19,036 DEBUG    - 훈련 데이터: 88,713행
2025-08-21 14:50:19,037 DEBUG    - 검증 데이터: 5,790행


🚀 통합 예측 시스템 학습 시작...

📊 damha_individual 모델 학습 중...
✅ damha_individual 학습 완료

📊 mirasia_individual 모델 학습 중...
✅ mirasia_individual 학습 완료

📊 main_restaurants 모델 학습 중...
✅ main_restaurants 학습 완료

📊 regular_outlets 모델 학습 중...
   📊 일반 업장 피처 생성:
      - 입력 데이터: 11546행
      - 느티나무 셀프BBQ 데이터: 11546행
      ✅ 10764개 시퀀스 생성
   📚 일반 업장 모델 학습:
      - 피처 수: 10764 x 31
      - 타겟 수: 10764 x 7
      - 매출 있는 비율: 80.9%
      ✅ Zero classifier 학습 완료
      ✅ Sales regressor 학습 완료
      ✅ 일자별 모델 7개 학습 완료
      ✅ 전체 학습 완료
✅ regular_outlets 학습 완료
   📊 일반 업장 피처 생성:
      - 입력 데이터: 690행
      - 느티나무 셀프BBQ 데이터: 690행
      ✅ 0개 시퀀스 생성

📊 special_venues 모델 학습 중...
   🎭 특수 업장 피처 생성:
      - 특수 업장 데이터: 21137행
      ✅ 19505개 시퀀스 생성
   🎭 특수 업장 모델 학습:
      - 피처 수: 19505 x 31
      - 타겟 수: 19505 x 7
      - 이벤트 있는 비율: 86.1%
      ✅ Event classifier 학습 완료


2025-08-21 14:51:52,292 DEBUG === 모델 학습 상태 확인 ===
2025-08-21 14:51:52,294 DEBUG damha_individual: 학습됨=True
2025-08-21 14:51:52,294 DEBUG mirasia_individual: 학습됨=True
2025-08-21 14:51:52,295 DEBUG main_restaurants: 학습됨=True
2025-08-21 14:51:52,295 DEBUG regular_outlets: 학습됨=True
2025-08-21 14:51:52,295 DEBUG special_venues: 학습됨=True
2025-08-21 14:51:52,297 DEBUG 
=== 테스트 파일: 10개 ===
2025-08-21 14:51:52,300 DEBUG TEST_08.csv: (5404, 3), 매출수량 범위: [0, 484]
2025-08-21 14:51:52,303 DEBUG TEST_09.csv: (5404, 3), 매출수량 범위: [0, 501]
2025-08-21 14:51:52,307 DEBUG TEST_02.csv: (5404, 3), 매출수량 범위: [-9, 407]


      ✅ Conditional regressor 학습 완료
      ✅ 연회장 전용 모델 학습 완료
      ✅ 라그로타 전용 모델 학습 완료
      ✅ 전체 학습 완료
✅ special_venues 학습 완료
   🎭 특수 업장 피처 생성:
      - 특수 업장 데이터: 1440행
      ✅ 0개 시퀀스 생성

📊 모델 성능 요약:
--------------------------------------------------
damha_individual     | SMAPE: ∞        | MAE: ∞        | 샘플:    0 | insufficient_validation_data
mirasia_individual   | SMAPE: ∞        | MAE: ∞        | 샘플:    0 | insufficient_validation_data
main_restaurants     | SMAPE: ∞        | MAE: ∞        | 샘플:    0 | insufficient_validation_data
regular_outlets      | SMAPE: ∞        | MAE: ∞        | 샘플:    0 | insufficient_validation_data
special_venues       | SMAPE: ∞        | MAE: ∞        | 샘플:    0 | insufficient_validation_data

🎉 전체 시스템 학습 완료!
통합 예측 시스템 예측 시작...
damha_individual 예측 중...
   업장: ['담하']
   데이터: 1176행
   damha_individual 예측 시도: 1176행
   피처 생성: (42, 40)
   예측 완료: (42, 7)
   성공: (42, 7)
mirasia_individual 예측 중...
   업장: ['미라시아']
   데이터: 868행
   mirasia_individual 예측 시도: 868행
 

...

   예측 완료: (49, 7)
   성공: (49, 7)
regular_outlets 예측 중...
   업장: ['느티나무 셀프BBQ']
   데이터: 644행
   regular_outlets 예측 시도: 644행
   📊 일반 업장 피처 생성:
      - 입력 데이터: 644행
      - 느티나무 셀프BBQ 데이터: 644행
      ✅ 23개 시퀀스 생성
   피처 생성: (23, 31)
   예측 완료: (23, 7)
   성공: (23, 7)
special_venues 예측 중...
   업장: ['연회장', '라그로타']
   데이터: 1344행
   special_venues 예측 시도: 1344행
   🎭 특수 업장 피처 생성:
      - 특수 업장 데이터: 1344행
      ✅ 48개 시퀀스 생성
   피처 생성: (48, 31)
   예측 완료: (48, 7)
   성공: (48, 7)
예측 완료! 성공한 카테고리: 5개
main_restaurants: 46개 음수값 제거됨
=== 예측 결과 상세 분석 ===
통합 예측 시스템 예측 시작...
damha_individual 예측 중...
   업장: ['담하']
   데이터: 1176행
   damha_individual 예측 시도: 1176행
   피처 생성: (42, 40)
   예측 완료: (42, 7)
   성공: (42, 7)
mirasia_individual 예측 중...
   업장: ['미라시아']
   데이터: 868행
   mirasia_individual 예측 시도: 868행
   피처 생성: (31, 20)
   예측 완료: (31, 7)
   성공: (31, 7)
main_restaurants 예측 중...
   업장: ['포레스트릿', '카페테리아', '화담숲주막', '화담숲카페']
   데이터: 1372행
   main_restaurants 예측 시도: 1372행
   피처 생성: (49, 59)
   예측 완료: (49, 7)
   성공: (49,

2025-08-21 14:51:53,133 DEBUG 
damha_individual:
2025-08-21 14:51:53,133 DEBUG   형태: (42, 7)
2025-08-21 14:51:53,134 DEBUG   범위: [0.40, 49.11]
2025-08-21 14:51:53,134 DEBUG   음수 개수: 0
2025-08-21 14:51:53,134 DEBUG   0값 개수: 0
2025-08-21 14:51:53,135 DEBUG   샘플 0: [2.3416035 2.1164613 2.4169135 2.5063384 2.1607053 2.6321082 2.03144  ]
2025-08-21 14:51:53,136 DEBUG   샘플 1: [20.233892 19.948633 20.453667 18.647081 19.428555 23.353422 27.455454]
2025-08-21 14:51:53,136 DEBUG   샘플 2: [16.915096 17.224459 19.124622 17.542297 20.445251 22.18529  26.38723 ]
2025-08-21 14:51:53,137 DEBUG 
mirasia_individual:
2025-08-21 14:51:53,137 DEBUG   형태: (31, 7)
2025-08-21 14:51:53,137 DEBUG   범위: [0.00, 164.92]
2025-08-21 14:51:53,137 DEBUG   음수 개수: 0
2025-08-21 14:51:53,138 DEBUG   0값 개수: 14
2025-08-21 14:51:53,138 DEBUG   샘플 0: [0.04902602 0.0401122  0.0401122  0.0401122  0.0401122  0.0401122
 0.04902602]
2025-08-21 14:51:53,139 DEBUG   샘플 1: [1.78227735 0.8383463  0.8383463  0.8383463  0.8383463  0.838

      ✅ 23개 시퀀스 생성
   피처 생성: (23, 31)
   예측 완료: (23, 7)
   성공: (23, 7)
special_venues 예측 중...
   업장: ['연회장', '라그로타']
   데이터: 1344행
   special_venues 예측 시도: 1344행
   🎭 특수 업장 피처 생성:
      - 특수 업장 데이터: 1344행
      ✅ 48개 시퀀스 생성
   피처 생성: (48, 31)
   예측 완료: (48, 7)
   성공: (48, 7)
예측 완료! 성공한 카테고리: 5개
main_restaurants: 46개 음수값 제거됨
제출 포맷 분석 완료:
   - 테스트 케이스: 10개
   - 업장-메뉴 조합: 193개
제출 파일 생성 중...
   처리 중: TEST_00 <- TEST_00.csv
통합 예측 시스템 예측 시작...
damha_individual 예측 중...
   업장: ['담하']
   데이터: 1176행
   damha_individual 예측 시도: 1176행
   피처 생성: (42, 40)
   예측 완료: (42, 7)
   성공: (42, 7)
mirasia_individual 예측 중...
   업장: ['미라시아']
   데이터: 868행
   mirasia_individual 예측 시도: 868행
   피처 생성: (31, 20)
   예측 완료: (31, 7)
   성공: (31, 7)
main_restaurants 예측 중...
   업장: ['포레스트릿', '카페테리아', '화담숲주막', '화담숲카페']
   데이터: 1372행
   main_restaurants 예측 시도: 1372행
   피처 생성: (49, 59)
   예측 완료: (49, 7)
   성공: (49, 7)
regular_outlets 예측 중...
   업장: ['느티나무 셀프BBQ']
   데이터: 644행
   regular_outlets 예측 시도: 644행
   📊 일반 업장 피처 생성:
  

2025-08-21 14:51:59,516 DEBUG 제출 파일 형태: (70, 194)
2025-08-21 14:51:59,518 DEBUG 0이 아닌 값 개수: 1922
2025-08-21 14:51:59,520 DEBUG 음수값 개수: 0
2025-08-21 14:51:59,520 DEBUG 
담하 관련 컬럼 수: 42
2025-08-21 14:51:59,521 DEBUG 담하 처음 5개 컬럼 값:
2025-08-21 14:51:59,521 DEBUG    담하_(단체) 공깃밥  담하_(단체) 생목살 김치전골 2.0  담하_(단체) 은이버섯 갈비탕  담하_(단체) 한우 우거지 국밥  \
0            0                     0                 0                  0   
1            0                     0                 0                  0   
2            0                     0                 0                  0   
3            0                     0                 0                  0   
4            0                     0                 0                  0   
5            0                     0                 0                  0   
6            0                     0                 0                  0   
7            0                     0                 0                  0   
8            0                     0             

제출 파일 생성 완료
